# Sesión 7 — Neo4j (grafos), curso 2026-27

![Logotipo de Neo4j](https://raw.githubusercontent.com/dsevilla/bdge/26-27/docs/img/neo4j-logo-notebook.png)

Esta sesión construye un grafo con los datos de Stack Overflow en español y lo consulta con Cypher desde Python, en Google Colab o con Docker Compose.

## Cómo se usa este boletín

Este boletín es **evaluable** y tiene dos funciones:

1. **Guía de estudio.** Los ejemplos y los primeros ejercicios recorren lo que hay que saber de Neo4j para el examen. Las transparencias **«Para el examen»** marcan, sección a sección, qué es lo importante y qué se suele preguntar.
2. **Resultado de entrega.** Los ejercicios 4 al 10, los de la sección «Ejercicios evaluables», se realizan en el orden indicado, y finalmente construirán la solución a entregar. Los parámetros que te corresponden se derivan de tu identificador. Al terminar, una celda lee el resultado final, otra lo envía al registro con el cliente result-only y una tercera permite consultar el ID y la fecha del último envío.

Puedes apoyarte en IA para resolver los ejercicios. El examen y la defensa individual se hacen **sin IA y sin este cuaderno**: lo que cuenta al final es que sepas explicar cada decisión.

### Identificación de la entrega

Escribe tu DNI o NIE en la celda siguiente, tal como figura en tu documento y con o sin la letra final. Su valor **no sale hacia el registro**: lo que se registra es una huella SHA-256 ligada al curso y al boletín. Aun así es un dato pseudonimizado, no anónimo, así que no publiques el cuaderno con tu DNI escrito.

El cuaderno sólo usa los dígitos: si tu identificador empieza por X, Y, Z o U, esa letra se sustituye por un dígito (X→0, Y→1, Z→2, U→3) y la letra final se descarta. Así, `12345678Z` y `12345678` son lo mismo, y `X1234567L` equivale a `01234567`. Tus parámetros salen de esos dígitos. Si el cuaderno rechaza tu identificador, avisa al profesor.

Esta celda sólo deja preparado el cuaderno: no descarga ni envía nada. El identificador se usa más adelante, al llegar a los **ejercicios evaluables**: allí se te vuelve a mostrar para que lo compruebes, se descarga el cliente de entregas, se calcula la huella y se derivan tus parámetros. El envío del resultado es el último paso del boletín.

In [ ]:
# El identificador sólo se usa aquí dentro; lo que se registra es su huella.
CURSO: str = "26-27"
DNI: str = "00000000T"  # Tu DNI o NIE, con o sin letra final
BOLETIN: str = "neo4j-sesion7"

In [ ]:
import sys

RunningInCOLAB: bool = "google.colab" in sys.modules

### Clave del túnel de Neo4j Browser

**Antes de ejecutar el cuaderno en Colab**, guarda la clave del túnel que se ha dado en clase en los *Secretos* de Colab: pulsa el icono de la llave (🔑) de la barra lateral izquierda, elige *Añadir nuevo secreto*, escribe `CLAVE_TUNEL` como nombre y la clave como valor, y activa el interruptor *Acceso desde el cuaderno*. Los secretos son tuyos y valen para todos tus cuadernos, así que sólo lo haces una vez y la clave no queda escrita en el cuaderno. La clave cambia cada curso y no se publica.

Sin ella, Neo4j se instala y el cuaderno funciona igual, pero no se abre el túnel y no podrás usar Neo4j Browser desde tu navegador. Con Docker Compose no hace falta: Neo4j Browser está en `http://localhost:7474`.

Si prefieres no usar los secretos, escribe la clave en `CLAVE_TUNEL` en la celda siguiente; en ese caso no publiques ni compartas el cuaderno con la clave escrita.

In [ ]:
# Clave del túnel que se ha dado en clase (cambia cada curso). Sólo hace falta en Colab.
# Se lee del secreto CLAVE_TUNEL de Colab; escríbela aquí sólo si no usas secretos.
CLAVE_TUNEL: str = ""

if RunningInCOLAB and not CLAVE_TUNEL:
    try:
        from google.colab import userdata
    except ImportError:
        pass  # El colab-check del CI simula Colab sin este paquete.
    else:
        try:
            CLAVE_TUNEL = userdata.get("CLAVE_TUNEL")
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass  # Sin secreto o sin acceso desde el cuaderno: se avisa a continuación.

if RunningInCOLAB and not CLAVE_TUNEL:
    import warnings

    warnings.warn(
        "No hay clave del túnel: añade el secreto CLAVE_TUNEL (icono de la llave, con el "
        "acceso desde el cuaderno activado) o escríbela en CLAVE_TUNEL y vuelve a ejecutar "
        "esta celda. Sin ella Neo4j funciona, pero no podrás abrir Neo4j Browser.",
        stacklevel=1,
    )

In [ ]:
if RunningInCOLAB:
    import urllib.request

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/dsevilla/bdge/refs/heads/26-27/addendum/frp-neo4j/setup-neo4j-colab.sh",
        "setup-neo4j-colab.sh",
    )

In [ ]:
import os
import subprocess

if RunningInCOLAB:
    URL_output: str = subprocess.run(
        ["sh", "./setup-neo4j-colab.sh"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        # El script lee la clave del túnel de esta variable de entorno.
        env={**os.environ, "BDGE_CLAVE_TUNEL": CLAVE_TUNEL},
    ).stdout

In [ ]:
if RunningInCOLAB:
    print(URL_output)

In [ ]:
import os
import re

BOLT_URL: str = f"bolt://{os.getenv('DB_HOSTNAME', 'localhost')}:7687"

if RunningInCOLAB:
    salida: str = URL_output

    if "[ERROR] Neo4j no respondió por Bolt" in salida:
        raise RuntimeError(
            "setup-neo4j-colab.sh no consiguió que Neo4j aceptase conexiones Bolt "
            "(ver el diagnóstico impreso en la celda anterior)."
        )

    # El script imprime líneas NEO4J_<NOMBRE>=<valor>; las de la traza de `sh -x` empiezan por "+".
    valores: dict[str, str] = dict(re.findall(r"^(NEO4J_[A-Z_]+)=(\S+)\s*$", salida, re.MULTILINE))
    # Neo4j arranca con una contraseña aleatoria; la conexión de más abajo la lee de aquí.
    os.environ["NEO4J_PASSWORD"] = valores["NEO4J_PASSWORD"]
    print(f"Contraseña de Neo4j (usuario neo4j): {os.environ['NEO4J_PASSWORD']}")

    if "NEO4J_HTTP_URL" in valores:
        print(f"URL de conexión al interfaz Web: {valores['NEO4J_HTTP_URL']}")
        print(f"URL de conexión de BOLT: {valores['NEO4J_BOLT_URL']}")
        print("Neo4j Browser pide el usuario neo4j y la contraseña anterior.")
    else:
        print(
            "Aviso: no se abrió el túnel frp (¿falta CLAVE_TUNEL o no es la del curso?); "
            "Neo4j Browser no será accesible desde fuera de esta máquina."
        )
        print("La conexión Bolt local que usa este cuaderno no se ve afectada.")

In [ ]:
%pip install neo4j-rust-ext fsspec pyarrow requests aiohttp 'pandas[parquet,performance]'

In [ ]:
# @title Diapositiva: El recorrido de Neo4j
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">El recorrido de Neo4j</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Crear un grafo con <em>constraints</em>, nodos y relaciones desde Parquet</li><li style=\'margin:10px 0;list-style-position:outside\'>Entender <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>UNWIND</code>, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> y la idempotencia de las cargas</li><li style=\'margin:10px 0;list-style-position:outside\'>Explorar patrones, recorridos, agregaciones y planes <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EXPLAIN</code>/<code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>PROFILE</code></li><li style=\'margin:10px 0;list-style-position:outside\'>Derivar relaciones para las preguntas frecuentes</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Idea central: el valor del grafo está en los recorridos que permite hacer de forma muy eficiente</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

In [ ]:
from pprint import pprint as pp

import pandas as pd

In [ ]:
# @title Diapositiva: Qué trae el dump para el grafo
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Qué trae el <em>dump</em> para el grafo</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Usuarios, publicaciones, comentarios, votos y etiquetas son posibles nodos</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Id</code> da una identidad estable; las referencias entre entidades sugieren relaciones</li><li style=\'margin:10px 0;list-style-position:outside\'>Propiedades opcionales pueden faltar y deben tratarse al cargar</li><li style=\'margin:10px 0;list-style-position:outside\'>El esquema Parquet será la guía para construir nodos y relaciones</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Antes de escribir Cypher, decidimos qué merece ser nodo, propiedad o relación</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

<details><summary><b>🗂️ Formato de los datos de Stack Overflow</b> <span style="color: #007acc;">▶️ Haz clic para ver el esquema completo</span></summary>

La descripción siguiente resume la [documentación vigente del *data dump* y de SEDE](https://meta.stackexchange.com/questions/2677/database-schema-documentation-for-the-public-data-dump-and-sede/2678). Las prácticas utilizan el *dump* de Stack Overflow en español; SEDE contiene además tablas que no se distribuyen en esos ocho ficheros XML.

> **Notas generales**
>
> - Cada fichero contiene elementos `<row ... />` codificados en UTF-8 y se distribuye dentro de un archivo 7z.
> - Los campos opcionales cuyo valor es nulo no aparecen como atributos en el XML. En los Parquet de la asignatura se representan normalmente como `null`.
> - Las fechas están en UTC y suelen usar el formato `YYYY-MM-DDTHH:mm:ss.fff`.
> - En esta sesión el grafo se construye sólo con `Posts`; `Users.parquet` se lee en un ejercicio evaluable y el resto de ficheros no se usa.
> - `Id` identifica la fila; los campos indicados como FK hacen referencia al `Id` de otra tabla.

---

:file_folder: <b>Badges.xml</b> — insignias concedidas a usuarios

- `Id`: clave primaria de la concesión.
- `UserId`: usuario que recibe la insignia; FK a `Users.Id`.
- `Name`: nombre de la insignia.
- `Date`: fecha de concesión.
- `Class`: clase de insignia (`1` = oro, `2` = plata, `3` = bronce).
- `TagBased`: `True` si está asociada a una etiqueta; `False` si es una insignia con nombre propio.

---

:file_folder: <b>Comments.xml</b> — comentarios en publicaciones

- `Id`: clave primaria.
- `PostId`: publicación comentada; FK a `Posts.Id`.
- `Score`: puntuación del comentario.
- `Text`: cuerpo del comentario.
- `CreationDate`: fecha de creación.
- `UserDisplayName`: nombre visible del autor cuando no puede referenciarse mediante `UserId`.
- `UserId`: autor; FK opcional a `Users.Id`, ausente si el usuario fue eliminado.
- `ContentLicense`: licencia Creative Commons del contenido.

---

:file_folder: <b>Posts.xml</b> — publicaciones no eliminadas

- `Id`: clave primaria.
- `PostTypeId`: tipo de publicación:
  - `1` pregunta; `2` respuesta; `3` wiki de etiqueta huérfana; `4` extracto de wiki de etiqueta; `5` wiki de etiqueta.
  - `6` candidatura a moderación; `7` marcador wiki; `8` wiki de privilegio; `9` artículo; `10` artículo de ayuda.
  - `12` colección; `13` respuesta a cuestionario de moderación; `14` anuncio; `15` discusión de colectivo; `17` colección de colectivo.
- `AcceptedAnswerId`: respuesta aceptada; FK opcional a `Posts.Id`, sólo para preguntas.
- `ParentId`: pregunta a la que responde; FK a `Posts.Id`, sólo para respuestas.
- `CreationDate`: fecha de creación.
- `Score`: puntuación neta.
- `ViewCount`: número de visitas, opcional.
- `Body`: cuerpo ya renderizado como HTML, no Markdown.
- `OwnerUserId`: autor; FK opcional a `Users.Id`. No aparece si se eliminó el usuario y vale `-1` en las entradas de wiki de etiqueta.
- `OwnerDisplayName`: nombre del autor cuando no hay un usuario referenciable.
- `LastEditorUserId`: último editor; FK opcional a `Users.Id`.
- `LastEditorDisplayName`: nombre del último editor cuando no hay un usuario referenciable.
- `LastEditDate`: fecha de la edición más reciente, opcional.
- `LastActivityDate`: fecha de la actividad más reciente.
- `Title`: título de la pregunta o, en determinados posts de tipo `4`/`5`, nombre de la etiqueta.
- `Tags`: etiquetas de la pregunta o etiqueta asociada a determinados posts de tipo `4`/`5`.
- `AnswerCount`: número de respuestas no eliminadas; sólo aparece en preguntas.
- `CommentCount`: número de comentarios, opcional.
- `FavoriteCount`: número histórico de favoritos, opcional.
- `ClosedDate`: fecha de cierre, sólo si la publicación está cerrada.
- `CommunityOwnedDate`: fecha desde la que la publicación es *community wiki*, si corresponde.
- `ContentLicense`: licencia Creative Commons del contenido.

Desde el *dump* de julio de 2025, `Posts.xml` incluye dos publicaciones artificiales con los identificadores `1000000001` y `1000000010`; no existen en SEDE y pueden excluirse. `DeletionDate` pertenece a `PostsWithDeleted` en SEDE y **no** aparece en el *data dump*.

---

:file_folder: <b>PostHistory.xml</b> — historial de las publicaciones

- `Id`: clave primaria del evento.
- `PostHistoryTypeId`: tipo de evento:
  - `1` título inicial; `2` cuerpo inicial; `3` etiquetas iniciales.
  - `4` edición de título; `5` edición de cuerpo; `6` edición de etiquetas.
  - `7`, `8`, `9`: reversión de título, cuerpo o etiquetas.
  - `10` cierre; `11` reapertura; `12` eliminación; `13` restauración; `14` bloqueo; `15` desbloqueo; `16` pasa a *community wiki*.
  - `17` migración antigua (sustituida por `35`/`36`); `18` fusión; `19` protección; `20` desprotección; `21` desasociación del autor; `22` deshacer fusión.
  - `24` edición sugerida aplicada; `25` publicación tuiteada; `31` discusión de comentarios movida al chat.
  - `33` aviso añadido; `34` aviso retirado; `35` migrada a otro sitio; `36` migrada desde otro sitio; `37` origen de fusión; `38` destino de fusión.
  - `50` subida por el usuario Community; `52` entrada en Hot Network Questions; `53` retirada de HNQ por moderación; `66` creada mediante el asistente de preguntas.
- `PostId`: publicación afectada; FK a `Posts.Id`.
- `RevisionGUID`: agrupa los registros producidos por una misma revisión.
- `CreationDate`: fecha del evento.
- `UserId`: usuario que realizó el cambio; FK opcional a `Users.Id`.
- `UserDisplayName`: nombre visible si el usuario fue eliminado o si se trata del autor de una publicación migrada.
- `Comment`: comentario de edición; para el tipo `10` contiene el motivo de cierre y para `33`/`34`, el identificador del aviso.
- `Text`: valor nuevo en crudo. En ciertos eventos de moderación contiene JSON con los votantes; en duplicados incluye `OriginalQuestionIds`, y para el tipo `17` contiene los detalles de migración.
- `ContentLicense`: licencia Creative Commons del contenido.

---

:file_folder: <b>PostLinks.xml</b> — enlaces entre publicaciones

- `Id`: clave primaria.
- `CreationDate`: fecha de creación del enlace.
- `PostId`: publicación origen; FK a `Posts.Id`.
- `RelatedPostId`: publicación relacionada o destino; FK a `Posts.Id`.
- `LinkTypeId`: tipo de enlace (`1` = enlazada, `3` = duplicada).

---

:file_folder: <b>Users.xml</b> — usuarios

- `Id`: clave primaria.
- `Reputation`: reputación.
- `CreationDate`: fecha de alta.
- `DisplayName`: nombre visible.
- `LastAccessDate`: última carga de una página por el usuario; se actualiza como máximo cada 30 minutos.
- `WebsiteUrl`: sitio web, opcional.
- `Location`: ubicación declarada, opcional.
- `AboutMe`: presentación del perfil, opcional.
- `Views`: visitas al perfil.
- `UpVotes`: votos positivos emitidos.
- `DownVotes`: votos negativos emitidos.
- `ProfileImageUrl`: URL de la imagen de perfil, opcional.
- `EmailHash`: actualmente siempre es nulo y, por tanto, no aparece en el XML.
- `AccountId`: identificador del perfil de red de Stack Exchange; opcional si el usuario oculta esa comunidad en su perfil.

El antiguo campo `Age` ya no forma parte del esquema documentado.

---

:file_folder: <b>Votes.xml</b> — votos y otras acciones sobre publicaciones

- `Id`: clave primaria.
- `PostId`: publicación afectada; FK a `Posts.Id`.
- `VoteTypeId`: tipo de voto o acción:
  - `-1` informar a moderación; `0` deshacer voto de moderación; `1` aceptación por el autor; `2` voto positivo; `3` voto negativo; `4` ofensivo.
  - `5` favorito/marcador (función retirada en octubre de 2022); `6` cierre (desde 2013 se guarda sólo en `PostHistory`); `7` reapertura.
  - `8` inicio de recompensa; `9` fin de recompensa; `10` eliminación; `11` restauración; `12` spam.
  - `15` revisión de moderación; `16` aprobación de edición sugerida; `17`–`21` reacciones/seguimiento; `22`–`28` reacciones de Teams.
  - `29` obsoleta; `30` no obsoleta; `31` voto previo; `32` voto positivo y `33` voto negativo en discusión de colectivo; `35`–`37` evaluación privada de respuesta de IA.
- `UserId`: usuario; sólo está presente para los tipos `5` y `8` (`-1` si se eliminó).
- `CreationDate`: fecha del voto. Por privacidad, la hora se elimina.
- `BountyAmount`: importe de la recompensa; sólo para los tipos `8` y `9`.

No todos los tipos enumerados pueden aparecer en SEDE o en el *dump* público; algunos son exclusivos de Teams o de experimentos concretos.

---

:file_folder: <b>Tags.xml</b> — etiquetas

- `Id`: clave primaria.
- `TagName`: nombre de la etiqueta.
- `Count`: número de preguntas que la usan.
- `ExcerptPostId`: publicación con el extracto de la wiki; FK opcional a `Posts.Id`.
- `WikiPostId`: publicación con el texto de la wiki; FK opcional a `Posts.Id`.
- `IsModeratorOnly`: indica si sólo moderación puede usarla.
- `IsRequired`: indica si su uso es obligatorio.

---

### Diagrama entidad-relación de SEDE

![Diagrama entidad-relación de Stack Exchange Data Explorer](https://raw.githubusercontent.com/dsevilla/bdge/26-27/docs/img/stackexchange-sede-erd.png)

El diagrama corresponde a **SEDE**, cuyo esquema es más rico que los ocho ficheros del *data dump*. Se incluye como visión global de tablas y relaciones; no implica que todas esas tablas estén disponibles en los Parquet de las prácticas. Fuente y autoría: [“The Schema Visualized”](https://meta.stackexchange.com/a/326361).

</details>

In [ ]:
# @title Diapositiva: Del esquema tabular al grafo
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Del esquema tabular al grafo</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Cargar Parquet por lotes, sin las columnas que no se usan</li><li style=\'margin:10px 0;list-style-position:outside\'>Conectar con Neo4j por Bolt desde Python con el cliente síncrono</li><li style=\'margin:10px 0;list-style-position:outside\'>Elegir etiquetas y propiedades que representen las entidades</li><li style=\'margin:10px 0;list-style-position:outside\'>Preparar restricciones antes de insertar nodos y relaciones</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>El grafo empieza con una identidad clara para cada entidad</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

Conexión síncrona con la base de datos:

In [ ]:
from neo4j import Driver, EagerResult, GraphDatabase, Record, basic_auth

cypher_query: str = """
MATCH (n)
RETURN n AS id
LIMIT 10;
"""

driver: Driver = GraphDatabase.driver(
    BOLT_URL, auth=basic_auth("neo4j", os.getenv("NEO4J_PASSWORD", ""))
)

result: EagerResult = driver.execute_query(cypher_query)
for record in result.records:
    print(record["id"])

### Partir de una base de datos vacía

El cuaderno reconstruye siempre el grafo desde cero. La carga crea los nodos con `CREATE` y, como `Post.Id` tendrá una restricción de unicidad, volver a cargar sobre una base ya llena fallaría en el primer lote al encontrar un `Id` repetido. Además, una carga interrumpida no se distinguiría de una completa. Por eso la celda siguiente borra todos los nodos y relaciones antes de empezar. La primera vez, y siempre en Colab, la base ya está vacía y no tarda nada. Las restricciones no se borran: se crean con `IF NOT EXISTS`.

El borrado va en lotes de 10 000 filas con `CALL { ... } IN TRANSACTIONS`. Un solo `DETACH DELETE` de todo el grafo sería una única transacción con cientos de miles de nodos y relaciones, que tendría que caber entera en la memoria de Neo4j. `CALL ... IN TRANSACTIONS` sólo puede ejecutarse en una transacción implícita (*auto-commit*): por eso usa `session.run()` y no `driver.execute_query()`, que ejecuta cada consulta dentro de una transacción explícita.

In [ ]:
from typing import LiteralString

from neo4j import Session


def clear_database() -> None:
    """Borra todos los nodos y relaciones, en transacciones de 10 000 filas."""
    query: LiteralString = """
    MATCH (n)
    CALL (n) {
      DETACH DELETE n
    } IN TRANSACTIONS OF 10000 ROWS
    """
    session: Session
    with driver.session() as session:
        session.run(query).consume()


clear_database()

nodos_restantes: int = driver.execute_query("MATCH (n) RETURN count(n) AS nodes").records[0][
    "nodes"
]
print(f"Nodos en la base de datos: {nodos_restantes}")
assert nodos_restantes == 0, "La base de datos no ha quedado vacía"

In [ ]:
# @title Diapositiva: Conexión y esquema
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Conexión y esquema</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Bolt permite enviar consultas desde el cliente Python</li><li style=\'margin:10px 0;list-style-position:outside\'>El esquema tabular aporta nombres, tipos y referencias</li><li style=\'margin:10px 0;list-style-position:outside\'>Una entidad puede convertirse en nodo, propiedad o relación</li><li style=\'margin:10px 0;list-style-position:outside\'>Las restricciones harán comprobable la identidad durante la carga</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Antes de insertar, fijamos qué no puede duplicarse</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Identidad e integridad antes de cargar
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Identidad e integridad antes de cargar</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Declarar unicidad para <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>User.Id</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Post.Id</code></li><li style=\'margin:10px 0;list-style-position:outside\'>Entender que cada <em>constraint</em> crea o usa un índice</li><li style=\'margin:10px 0;list-style-position:outside\'>Hacer que <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> pueda encontrar la identidad de forma eficiente</li><li style=\'margin:10px 0;list-style-position:outside\'>Separar restricciones de identidad y relaciones que aún construiremos</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La integridad no es un paso posterior: condiciona cómo cargamos el grafo</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### Creación de Constraints (Restricciones de Integridad)

Antes de cargar los datos, es una **buena práctica** crear constraints de unicidad en los atributos que identifican de manera única a cada nodo. Los constraints no solo aseguran la integridad de los datos (evitando duplicados), sino que también crean automáticamente **índices** que aceleran enormemente las operaciones `MERGE` y las búsquedas.

En nuestro modelo de datos de StackOverflow:
- Cada `User` se identifica de forma única por su `Id`
- Cada `Post` (que puede ser `Question` o `Answer`) se identifica de forma única por su `Id`

Crear estos constraints **antes** de la carga masiva de datos mejorará significativamente el rendimiento.

In [ ]:
# Crear constraint de unicidad para User.Id
# Esto crea automáticamente un índice en User.Id
driver.execute_query("""
    CREATE CONSTRAINT user_id_unique IF NOT EXISTS
    FOR (u:User) REQUIRE u.Id IS UNIQUE
    """)

In [ ]:
# Crear constraint de unicidad para Post.Id
# Las preguntas y las respuestas llevan también la etiqueta Post: comparten este índice
driver.execute_query("""
    CREATE CONSTRAINT post_id_unique IF NOT EXISTS
    FOR (p:Post) REQUIRE p.Id IS UNIQUE
    """)

Con estas restricciones, Neo4j garantiza que no puede haber dos nodos `User` ni dos nodos `Post` con el mismo `Id`, y el índice que crea cada una hace que `MERGE` y las búsquedas por `Id` no tengan que recorrer todos los nodos de la etiqueta.

In [ ]:
from asyncio import Task, create_task, to_thread
from collections.abc import Collection, Iterator
from concurrent.futures import FIRST_COMPLETED, Future, ThreadPoolExecutor, wait
from typing import Any, BinaryIO, LiteralString, cast

import fsspec
import pyarrow as pa
import pyarrow.parquet as pq

from neo4j import Driver, EagerResult

# Un hilo lee Parquet; estos tres envían lotes a Neo4j en paralelo.
BATCH_WRITERS: int = 3
PARQUET_BASE_URL: str = (
    "https://github.com/dsevilla/bd2-data/releases/download/es.stackoverflow.data-26-27"
)


def load_parquet_neo4j(
    tag: str,
    parquet_file: str,
    driver: Driver,
    batch_size: int = 10_000,
    exclude_columns: Collection[str] = (),
) -> int:
    """Carga un Parquet por lotes, sin las columnas excluidas, y cuenta los nodos creados."""
    if batch_size <= 0:
        raise ValueError("batch_size debe ser positivo")

    cypher_query: LiteralString = cast(
        LiteralString,
        f"""
        UNWIND $batch AS node
        CREATE (p:{tag})
        SET p = node
        """,
    )
    url: str = f"{PARQUET_BASE_URL}/{parquet_file}"
    expected_rows: int = 0
    nodes_created: int = 0
    pending: set[Future[EagerResult]] = set()
    done: set[Future[EagerResult]]
    errors: list[Exception] = []

    def _count_created(futures: set[Future[EagerResult]]) -> int:
        created: int = 0
        for future in futures:
            try:
                created += future.result().summary.counters.nodes_created
            except Exception as exc:
                errors.append(exc)
        return created

    pool: ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=BATCH_WRITERS) as pool:
        try:
            parquet_stream: BinaryIO
            with fsspec.open(url, "rb") as parquet_stream:
                parquet: pq.ParquetFile = pq.ParquetFile(parquet_stream)
                expected_rows = parquet.metadata.num_rows
                # PyArrow sólo decodifica las columnas que vamos a guardar en el grafo.
                columns: list[str] = [
                    name for name in parquet.schema_arrow.names if name not in exclude_columns
                ]
                batches: Iterator[pa.RecordBatch] = parquet.iter_batches(
                    batch_size=batch_size, columns=columns, use_threads=True
                )
                for record_batch in batches:
                    # Un diccionario por nodo. Sólo se convierte una columna cada vez.
                    batch: list[dict[str, Any]] = [{} for _ in range(record_batch.num_rows)]
                    for name in columns:
                        column: pa.Array = record_batch.column(name)
                        for row, value in zip(batch, column.to_pylist(), strict=True):
                            if value is not None:
                                row[name] = value
                    pending.add(pool.submit(driver.execute_query, cypher_query, batch=batch))

                    # Limitar también los lotes en espera: cada Future retiene
                    # sus diccionarios hasta que termina la consulta.
                    if len(pending) < BATCH_WRITERS:
                        continue
                    done, pending = wait(pending, return_when=FIRST_COMPLETED)
                    nodes_created += _count_created(done)
                    if errors:
                        break
        except Exception as exc:
            errors.append(exc)
        finally:
            # Esperar a todos los lotes ya enviados y conservar todos sus errores.
            nodes_created += _count_created(pending)

    if errors:
        raise ExceptionGroup(f"Falló la carga de {parquet_file}", errors)
    assert nodes_created == expected_rows, (
        f"{parquet_file}: {expected_rows} filas en Parquet y {nodes_created} nodos creados"
    )
    return nodes_created

Los posts se cargan en segundo plano. Los hilos solapan la lectura del Parquet y las escrituras por lotes. El paquete `neo4j-rust-ext` instala el driver oficial de Neo4j y acelera la codificación de los datos para Bolt.

Cada fila se convierte en un mapa de propiedades. El cargador rellena los mapas columna a columna y omite sólo los valores `None`. Así se conservan el cero y la cadena vacía, y una propiedad sin valor queda ausente en el nodo.

Se conservan todas las columnas de `Posts.parquet` salvo `Body`, el cuerpo HTML de cada publicación. Esta columna ocupa el 96 % del Parquet sin comprimir. Al seleccionar las demás columnas, PyArrow evita decodificar el HTML.

In [ ]:
load_posts_task: Task[int] = create_task(
    to_thread(load_parquet_neo4j, "Post", "Posts.parquet", driver, exclude_columns={"Body"})
)

In [ ]:
# @title Diapositiva: Preparar la carga y las relaciones
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Preparar la carga y las relaciones</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Los <em>constraints</em> fijan la identidad antes de leer Parquet</li><li style=\'margin:10px 0;list-style-position:outside\'>La carga por lotes mantiene acotada la memoria y la presión sobre Neo4j</li><li style=\'margin:10px 0;list-style-position:outside\'>Las relaciones se crean después de tener nodos localizables</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> permite repetir la creación de relaciones sin duplicarlas</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Con los posts ya cargados, falta preguntarles y completar el grafo con Cypher</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Para el examen: modelar un grafo
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #8a6a00;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#8a6a00" size="8" noshade style="display:none"><div style="color:#8a6a00;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#8a6a00" style="color:inherit"><b>Para el examen</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Para el examen: modelar un grafo</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Decidir qué es nodo, qué es propiedad y qué es relación es el modelado</li><li style=\'margin:10px 0;list-style-position:outside\'>La identidad se declara con una restricción de unicidad, que además crea un índice</li><li style=\'margin:10px 0;list-style-position:outside\'>Una relación tiene tipo y dirección, aunque al consultar se pueda ignorar la dirección</li><li style=\'margin:10px 0;list-style-position:outside\'>Neo4j no admite propiedades nulas: la ausencia se representa no enviando la clave</li><li style=\'margin:10px 0;list-style-position:outside\'>Se modela para el recorrido que se va a hacer, no para normalizar</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #8a6a00;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pregunta típica: pasar un esquema relacional a nodos y relaciones, y justificarlo</div></blockquote><hr color="#8a6a00" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Hablar con el grafo: Cypher
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Hablar con el grafo: Cypher</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Expresar patrones de nodos y relaciones con una sintaxis declarativa</li><li style=\'margin:10px 0;list-style-position:outside\'>Separar <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MATCH</code>, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>WHERE</code>, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>WITH</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>RETURN</code> para razonar por etapas</li><li style=\'margin:10px 0;list-style-position:outside\'>Usar parámetros en vez de construir texto de consulta</li><li style=\'margin:10px 0;list-style-position:outside\'>Preparar las operaciones que convertirán columnas en recorridos</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Cypher describe el camino que queremos encontrar, no una secuencia de bucles</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### El lenguaje Cypher

Cypher describe **patrones**: los nodos van entre paréntesis, `(u:User)`, y las relaciones entre corchetes con una flecha, `-[:WROTE]->`. La consulta dice qué forma tiene lo que se busca y el planificador decide cómo recorrerlo. La [*Cypher Cheat Sheet*](https://neo4j.com/docs/cypher-cheat-sheet/current/) resume toda la sintaxis.

Las celdas siguientes completan el grafo con Cypher. Los nodos `Post` ya cargados reciben su segunda etiqueta, `Question` o `Answer`, según `PostTypeId`; los demás tipos de publicación se quedan sólo con `Post`. De las propiedades `OwnerUserId` y `ParentId` salen después los usuarios y las relaciones `:WROTE` y `:ANSWERS`.

Primero se espera a que termine la carga. Después, todas las preguntas reciben la etiqueta `Question`:

In [ ]:
posts_cargados: int = await load_posts_task
print(f"Posts cargados: {posts_cargados:,}")

In [ ]:
result: EagerResult = driver.execute_query("""
    MATCH (n:Post {PostTypeId : 1})
      SET n:Question;
    """)
pp(result.summary.counters)

A todas las respuestas se las etiqueta con `Answer`.

In [ ]:
result: EagerResult = driver.execute_query("""
      MATCH (n:Post {PostTypeId : 2})
        SET n:Answer;
      """)
pp(result.summary.counters)

Se crea un nodo usuario (o se utiliza uno si ya existe) usando `OwnerUserId`, siempre que no esté vacío. También se usa `MERGE` para la relación: de ese modo se puede volver a ejecutar la celda sin duplicarla. La fecha de creación se añade después con `SET`, porque no forma parte de la identidad del patrón.

In [ ]:
result: EagerResult = driver.execute_query("""
    MATCH (n:Post)
    WHERE n.OwnerUserId IS NOT NULL AND n.OwnerUserId > 0
        MERGE (u:User {Id: n.OwnerUserId})
        MERGE (u)-[r:WROTE]->(n)
        SET r.CreationDate = n.CreationDate;
    """)
pp(result.summary.counters)

In [ ]:
query = """
MATCH r=(u:User)-[:WROTE]->(p:Post) RETURN r LIMIT 20
"""

result: EagerResult = driver.execute_query(query)
for record in result.records:
    pp(record["r"])

Añadimos una relación entre las preguntas y las respuestas. La pregunta se busca como `(q:Question:Post {Id: a.ParentId})`, con las dos etiquetas, aunque `:Question` bastaría para encontrarla.

En Neo4j cada índice pertenece a una etiqueta: la restricción de unicidad creó un índice sobre `:Post(Id)`, no sobre `:Question(Id)`. El planificador sólo tiene en cuenta los índices de las etiquetas escritas en el patrón, así que con `(q:Question {Id: a.ParentId})` no puede buscar en el índice la pregunta de cada respuesta, aunque el nodo también lleve la etiqueta `Post`. Lo que hace entonces es recorrer todas las preguntas y todas las respuestas y emparejarlas en una tabla *hash* (`ValueHashJoin`) que guarda en memoria todas las preguntas. Con `:Post` en el patrón, cada respuesta localiza su pregunta en el índice (`NodeUniqueIndexSeek`), sin esa tabla. Lo comprobarás con `EXPLAIN` en un ejercicio más adelante.

Los dos `MATCH` van separados a propósito. Escritos en uno solo, con una coma, el planificador puede preferir otro plan si sus estadísticas aún no reflejan la carga recién hecha.

In [ ]:
result: EagerResult = driver.execute_query("""
    MATCH (a:Answer)
    MATCH (q:Question:Post {Id: a.ParentId})
    MERGE (a)-[:ANSWERS]->(q)
    """)
pp(result.summary.counters)

Las consultas retornan resultados de los que se puede obtener un `dataframe` de `pandas` de forma muy sencilla.

In [ ]:
query = "MATCH q=(r)-[:ANSWERS]->(p) RETURN p.Id,r.Id;"

result: EagerResult = driver.execute_query(query)
df = pd.DataFrame(result.records, columns=result.keys)

df["r.Id"] = pd.to_numeric(df["r.Id"], downcast="unsigned")
df["p.Id"] = pd.to_numeric(df["p.Id"], downcast="unsigned")

df.plot(kind="scatter", x="p.Id", y="r.Id", figsize=(15, 15))

In [ ]:
# @title Diapositiva: Del lote de posts a un subgrafo
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Del lote de posts a un subgrafo</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>WROTE</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ANSWERS</code> se derivan de <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>OwnerUserId</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ParentId</code> con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MATCH</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code></li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>CREATE</code> es directo; <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> hace repetible una carga</li><li style=\'margin:10px 0;list-style-position:outside\'>Las relaciones necesitan nodos con una identidad ya garantizada</li><li style=\'margin:10px 0;list-style-position:outside\'>Una propiedad ausente no debe enviarse como si fuera un valor</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Los posts ya son nodos; ahora convertiremos sus etiquetas en estructura compartida</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Para el examen: cargar sin duplicar
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #8a6a00;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#8a6a00" size="8" noshade style="display:none"><div style="color:#8a6a00;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#8a6a00" style="color:inherit"><b>Para el examen</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Para el examen: cargar sin duplicar</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>CREATE</code> siempre crea; <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> busca el patrón y sólo crea si no lo encuentra</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> empareja el patrón <strong>completo</strong>: cuantas más propiedades lleve, menos empareja</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ON CREATE SET</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ON MATCH SET</code> separan el alta de la actualización</li><li style=\'margin:10px 0;list-style-position:outside\'>Una carga idempotente se puede repetir sin duplicar: es propiedad del modelo, no del script</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #8a6a00;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pregunta típica: dado un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>CREATE</code>, decir qué duplica y reescribirlo con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code></div></blockquote><hr color="#8a6a00" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Convertir Tags en relaciones
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Convertir <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Tags</code> en relaciones</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Separar una cadena como <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>&lt;python&gt;&lt;sql&gt;</code> en nombres de etiqueta</li><li style=\'margin:10px 0;list-style-position:outside\'>Crear nodos <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Tag</code> compartidos mediante <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code></li><li style=\'margin:10px 0;list-style-position:outside\'>Enlazar preguntas y etiquetas con una relación explícita</li><li style=\'margin:10px 0;list-style-position:outside\'>Decidir si mantener una o dos direcciones de la relación</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La representación elegida debe servir a los recorridos que preguntaremos</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## De una cadena de etiquetas a un subgrafo

`Tags` llega como una cadena como `<python><sql>`. En un grafo interesa convertir cada etiqueta en un nodo compartido. La restricción hace explícita su identidad y acelera `MERGE`; las dos direcciones de relación se mantienen aquí deliberadamente para poder discutir si esa redundancia compensa según las consultas.

Para crear un nodo por etiqueta hay que convertir esa cadena en una lista y la lista en filas, y de eso se ocupa `UNWIND`. Lo introducimos aquí, con la consulta que lo necesita.

In [ ]:
# @title Diapositiva: UNWIND: de una lista a filas
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px"><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>UNWIND</code>: de una lista a filas</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Desdoblar una lista en una fila por cada elemento</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>split()</code> convierte la cadena <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Tags</code> en una lista; <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>UNWIND</code> la recorre</li><li style=\'margin:10px 0;list-style-position:outside\'>Las cláusulas siguientes se ejecutan una vez por fila: un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> de <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Tag</code> y otro de su relación</li><li style=\'margin:10px 0;list-style-position:outside\'>La misma idea sirve para una lista de mapas que llega desde Python</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Una propiedad con varios valores en una cadena se convierte en filas y, después, en estructura</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### La cláusula UNWIND

`UNWIND` toma una lista y la "desdobla" en filas individuales. Es similar a un "for each" pero integrado en la consulta Cypher: cada elemento de la lista produce una fila, y las cláusulas que siguen (`MATCH`, `MERGE`, `RETURN`...) se ejecutan una vez por fila. Esto es especialmente útil para:
1. Recorrer una lista guardada en una propiedad, como las etiquetas de una pregunta
2. Crear varios nodos o relaciones a partir de una sola fila de entrada
3. Transformar datos estructurados en formato de lista

Veamos algunos ejemplos prácticos:

In [ ]:
# Ejemplo 1: UNWIND básico - desdoblar una lista simple
query_unwind_basico = """
UNWIND [1, 2, 3, 4, 5] AS number
RETURN number
"""

result = driver.execute_query(query_unwind_basico)
print("Ejemplo 1: Lista simple desdoblada")
for record in result.records:
    print(f"  Número: {record['number']}")

In [ ]:
# Ejemplo 2: UNWIND con diccionarios (mapas)
query_unwind_mapas = """
UNWIND [
    {name: 'Alice', age: 30},
    {name: 'Bob', age: 25},
    {name: 'Charlie', age: 35}
] AS person
RETURN person.name AS name, person.age AS age
"""

result = driver.execute_query(query_unwind_mapas)
print("\nEjemplo 2: Lista de diccionarios desdoblada")
for record in result.records:
    print(f"  {record['name']} tiene {record['age']} años")

In [ ]:
# Ejemplo 3: de una cadena de etiquetas a una fila por etiqueta
query_unwind_tags = """
WITH '<python><sql><neo4j>' AS tags
UNWIND split(substring(tags, 1, size(tags) - 2), '><') AS tag_name
RETURN tag_name
"""

result = driver.execute_query(query_unwind_tags)
print("\nEjemplo 3: Cadena de etiquetas desdoblada")
for record in result.records:
    print(f"  Etiqueta: {record['tag_name']}")

**`UNWIND` dentro de una consulta y desde Python**

En la consulta siguiente, `split()` trocea cada `q.Tags` en una lista y `UNWIND` produce una fila por etiqueta. Después, `MERGE (t:Tag {TagName: tag_name})` crea el nodo de la etiqueta la primera vez que aparece y lo reutiliza las siguientes, y el segundo `MERGE` crea la relación con la pregunta. Una lista vacía no produce ninguna fila: la consulta no sigue para esa pregunta.

El `WHERE` deja fuera las preguntas sin etiquetas. Una etiqueta ocupa al menos tres caracteres (`<a>`), así que `size(q.Tags) > 2` descarta la cadena vacía, para la que `substring` recibiría una longitud negativa; `IS NOT NULL` hace explícito el caso de la propiedad ausente.

La misma idea sirve cuando la lista no está en el grafo sino en Python: se pasa como parámetro (`UNWIND $batch AS fila`) y se procesa en una sola transacción, en lugar de ejecutar una consulta por registro. Es lo que hace por dentro `load_parquet_neo4j`, y se usa de nuevo en los ejercicios que actualizan nodos por lotes.

In [ ]:
driver.execute_query("""
CREATE CONSTRAINT tag_name_unique IF NOT EXISTS
FOR (t:Tag) REQUIRE t.TagName IS UNIQUE
""")
result: EagerResult = driver.execute_query("""
MATCH (q:Question)
WHERE q.Tags IS NOT NULL AND size(q.Tags) > 2
UNWIND split(substring(q.Tags, 1, size(q.Tags) - 2), '><') AS tag_name
MERGE (t:Tag {TagName: tag_name})
MERGE (q)-[:TAGGED_WITH]->(t)
MERGE (t)-[:TAGS]->(q)
""")
pp(result.summary.counters)

In [ ]:
# @title Diapositiva: Del esquema a los patrones
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Del esquema a los patrones</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Las etiquetas y relaciones convierten columnas en caminos</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>split()</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>UNWIND</code> convierten una cadena en una lista y la lista en filas</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> evita duplicar identidades al repetir la carga</li><li style=\'margin:10px 0;list-style-position:outside\'>Las etiquetas de <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Question</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Answer</code> permiten consultas más legibles</li><li style=\'margin:10px 0;list-style-position:outside\'>Ya podemos recorrer el grafo y contar lo que encontramos</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La estructura sólo cobra sentido cuando la usamos para preguntar</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Para el examen: de una lista a filas
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #8a6a00;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#8a6a00" size="8" noshade style="display:none"><div style="color:#8a6a00;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#8a6a00" style="color:inherit"><b>Para el examen</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Para el examen: de una lista a filas</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>UNWIND</code> convierte una lista en filas: una por elemento, y lo que sigue se ejecuta por fila</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>split()</code>, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>UNWIND</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> pasan una cadena de valores a nodos compartidos</li><li style=\'margin:10px 0;list-style-position:outside\'>Una lista vacía no produce filas: la consulta no continúa para ese registro</li><li style=\'margin:10px 0;list-style-position:outside\'>Con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>UNWIND &#36;batch</code>, una lista de mapas desde Python se procesa en una petición por lote, no por fila</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #8a6a00;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pregunta típica: dada una propiedad con varios valores en una cadena, escribir la consulta que crea un nodo por valor</div></blockquote><hr color="#8a6a00" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Explorar patrones del grafo
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Explorar patrones del grafo</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Seleccionar nodos por etiqueta y propiedades</li><li style=\'margin:10px 0;list-style-position:outside\'>Navegar relaciones de autoría y respuestas</li><li style=\'margin:10px 0;list-style-position:outside\'>Agregar preguntas, respuestas y etiquetas</li><li style=\'margin:10px 0;list-style-position:outside\'>Comparar <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EXPLAIN</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>PROFILE</code> con el trabajo real</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Primero buscamos; después mediremos cómo encuentra Neo4j el patrón</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## Explorando el Grafo con Consultas Básicas

Ahora que hemos cargado los datos en Neo4j, es momento de aprender a explorarlos. Esta sección te guiará a través de las consultas fundamentales de Cypher, desde las más simples hasta patrones más sofisticados.

### 1. Consultas Básicas de Selección

Empecemos con las consultas más básicas para recuperar nodos y entender la estructura del grafo.

Contar cuántos nodos de cada tipo tenemos:

In [ ]:
query = """
MATCH (n)
RETURN labels(n) AS node_labels, count(*) AS num_nodes
ORDER BY num_nodes DESC
"""

result: EagerResult = driver.execute_query(query)
print("Resumen de nodos en el grafo:")
for r in result.records:
    print(f"  {r['node_labels']}: {r['num_nodes']} nodos")

Obtener las 5 preguntas con mayor puntuación:

In [ ]:
query = """
MATCH (q:Question)
RETURN q.Id AS id, q.Title AS title, q.Score AS score
ORDER BY q.Score DESC
LIMIT 5
"""

result: EagerResult = driver.execute_query(query)
print("\nLas 5 preguntas con mayor puntuación:")
for r in result.records:
    print(f"  [{r['id']}] {r['title']} (Score: {r['score']})")

### Cómo se recogen los resultados de los ejercicios

Lo importante de cada ejercicio es la consulta. Cuando un ejercicio pide un resultado, no hay que extraer nada de él: se guarda **tal cual** lo que devuelve `execute_query`, en la variable que dice el enunciado.

```python
ejemplo: EagerResult = driver.execute_query("""
MATCH (q:Question)
RETURN q.Id, q.Score
LIMIT 3
""")

for identificador, puntuacion in ejemplo.records:
    print(identificador, puntuacion)
```

La comprobación saca de ahí lo que necesita. Para que pueda hacerlo, el `RETURN` debe devolver **las columnas que pide el enunciado, en ese orden**; los nombres (`AS`) los eliges tú.

Si quieres mirar el resultado, `ejemplo.records` es la lista de filas y cada fila se comporta como una tupla. No recorras `ejemplo` directamente: sus tres elementos son las filas (`records`), el resumen (`summary`) y los nombres de las columnas (`keys`).

### EJERCICIO 1: las respuestas mejor valoradas

Obtén las 5 respuestas (`:Answer`) con mayor `Score`, de mayor a menor, con dos columnas: su `Id` y su `Score`. Guarda el resultado de la consulta en `resultado_1`.

In [ ]:
# Comprobación del EJERCICIO 1: las respuestas mejor valoradas
assert "resultado_1" in globals(), "Guarda lo que devuelve execute_query en resultado_1"
assert len(resultado_1.keys) == 2, "Devuelve dos columnas: Id y Score"
assert len(resultado_1.records) == 5, "Se piden exactamente 5 respuestas"

esperado: list[Record] = driver.execute_query(
    "MATCH (a:Answer) RETURN a.Id, a.Score ORDER BY a.Score DESC LIMIT 5"
).records
assert [score for _, score in resultado_1.records] == [score for _, score in esperado], (
    "Las puntuaciones no son las de las cinco respuestas mejor valoradas "
    "(las columnas van en este orden: Id, Score)"
)
print("EJERCICIO 1: comprobado")

### 2. Consultas con Relaciones

Ahora veamos cómo navegar por las relaciones del grafo.

Encontrar qué usuario hizo una pregunta específica. El `Id` no se escribe dentro del texto de la consulta: va como **parámetro**, `$question_id`, y su valor se pasa a `execute_query` como argumento con nombre (`question_id=6`). El driver lo envía aparte del texto, así que no hay que construir la consulta con *f-strings* ni preocuparse de las comillas, y Neo4j reutiliza el mismo plan con otro valor. Los ejercicios siguientes usan parámetros de la misma forma:

In [ ]:
query = """
MATCH (u:User)-[:WROTE]->(q:Question)
WHERE q.Id = $question_id
RETURN u.Id AS user_id, q.Title AS question
"""

result: EagerResult = driver.execute_query(query, question_id=6)
print("\nUsuario que hizo la pregunta ID=6:")
for r in result.records:
    print(f"  Usuario {r['user_id']} preguntó: '{r['question']}'")

Encontrar respuestas a una pregunta, con el mismo parámetro:

In [ ]:
query = """
MATCH (q:Question {Id: $question_id})<-[:ANSWERS]-(a:Answer)
RETURN q.Title AS question, a.Id AS answer_id, a.Score AS score
ORDER BY a.Score DESC
"""

result: EagerResult = driver.execute_query(query, question_id=6)
print("\nRespuestas a la pregunta ID=6:")
for r in result.records:
    print(f"  Respuesta {r['answer_id']} (Score: {r['score']})")

### EJERCICIO 2: autor y número de respuestas en una sola consulta

Para la pregunta con `Id: 6`, obtén en una sola consulta una fila con dos columnas: el `Id` del usuario que la escribió (relación `:WROTE`) y el número de respuestas que tiene (relación `:ANSWERS`). Guarda el resultado de la consulta en `resultado_2`. Puedes escribir el `Id` en la consulta o pasarlo como parámetro, como en los ejemplos anteriores.

In [ ]:
# Comprobación del EJERCICIO 2: autor y número de respuestas en una sola consulta
assert "resultado_2" in globals(), "Guarda lo que devuelve execute_query en resultado_2"
assert len(resultado_2.records) == 1 and len(resultado_2.keys) == 2, (
    "Se pide una sola fila con dos columnas: autor y número de respuestas"
)
autor: int
respuestas: int
autor, respuestas = resultado_2.records[0]

# Camino paralelo: una consulta por el autor y otra, distinta, por el recuento.
autor_esperado: int = driver.execute_query(
    "MATCH (u:User)-[:WROTE]->(:Question {Id: 6}) RETURN u.Id"
).records[0][0]
respuestas_esperadas: int = driver.execute_query(
    "MATCH (:Answer)-[:ANSWERS]->(:Question {Id: 6}) RETURN count(*)"
).records[0][0]
assert autor == autor_esperado, "El autor de la pregunta 6 no coincide (es la primera columna)"
assert respuestas == respuestas_esperadas, "El número de respuestas no coincide"
print("EJERCICIO 2: comprobado")

### 3. Agregaciones y Estadísticas

Las funciones de agregación nos permiten calcular métricas sobre los datos del grafo.

Usuarios más activos (con más preguntas):

In [ ]:
query = """
MATCH (u:User)-[:WROTE]->(q:Question)
RETURN u.Id AS user_id, count(q) AS num_questions
ORDER BY num_questions DESC
LIMIT 10
"""

result: EagerResult = driver.execute_query(query)
print("\nTop 10 usuarios con más preguntas:")
for i, r in enumerate(result.records, 1):
    print(f"  {i}. Usuario {r['user_id']}: {r['num_questions']} preguntas")

In [ ]:
# @title Diapositiva: Leer un EXPLAIN
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Leer un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EXPLAIN</code></h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>El plan es un árbol: <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ProduceResults</code> es la raíz y los operadores de acceso a datos, las hojas</li><li style=\'margin:10px 0;list-style-position:outside\'>Se lee de abajo arriba: cada operador pasa sus filas al de encima</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>operatorType</code> dice qué hace: <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>NodeIndexSeek</code> usa un índice, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>NodeByLabelScan</code> recorre la etiqueta</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>args</code> indica sobre qué actúa cada operador (<code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Details</code>) y cuántas filas se esperan (<code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EstimatedRows</code>)</li><li style=\'margin:10px 0;list-style-position:outside\'>No ejecuta la consulta: las filas son estimaciones y no hay coste medido</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Primero se mira el operador de la hoja: ahí se ve si el índice sirve</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

Ahora con `EXPLAIN`, que muestra el plan que Neo4j ha elegido sin ejecutar la consulta. `summary.plan` es un diccionario de Python que representa el plan como un **árbol**: cada operador es un diccionario y sus operadores hijos van anidados en `children`. Al leer la salida de la celda siguiente, estos son los campos que importan:

- **Forma del árbol.** El diccionario más externo es la raíz (`ProduceResults`, el último paso, el que entrega el resultado). Siguiendo `children` se baja hasta las hojas, que son los operadores que leen del almacenamiento. La ejecución va al revés: empieza en las hojas y las filas suben hacia la raíz.
- **`operatorType`.** El nombre del operador. Mira primero cómo se localizan los nodos, en las hojas: `NodeUniqueIndexSeek` y `NodeIndexSeek` usan un índice; `NodeByLabelScan` recorre todos los nodos de una etiqueta y `AllNodesScan`, todos los del grafo. Más arriba aparecen `Expand(All)` (sigue relaciones), `Filter`, `EagerAggregation` (agrega, por ejemplo `count`) y `Sort` o `Top`. Puede llevar detrás el nombre de la base de datos (`@neo4j`).
- **`args`.** Argumentos del operador. `Details` indica sobre qué patrón o predicado actúa (por ejemplo, `t.TagName = 'java'`) y `EstimatedRows` es el número de filas que el planificador espera que salgan de él.
- **`identifiers`.** Las variables de la consulta disponibles en ese punto del plan: sirven para seguir qué operador produce cada `u` o `q`.

Como `EXPLAIN` no ejecuta nada, el plan sólo trae estimaciones, basadas en las estadísticas de la base de datos; es una forma barata de ver si una consulta usará un índice antes de lanzarla.

In [ ]:
query = """
EXPLAIN
MATCH (u:User)-[:WROTE]->(q:Question)
RETURN u.Id AS user_id, count(q) AS num_questions
ORDER BY num_questions DESC
LIMIT 10
"""

result: EagerResult = driver.execute_query(query)
print("\nPlan de ejecución para la consulta de usuarios con más preguntas:")
pp(result.summary.plan)

In [ ]:
# @title Diapositiva: Leer un PROFILE
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Leer un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>PROFILE</code></h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Tiene el mismo árbol que <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EXPLAIN</code>, pero ejecuta la consulta y mide el trabajo real</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>rows</code> son las filas que salen de un operador: se comparan con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EstimatedRows</code></li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>dbHits</code> son los accesos al almacenamiento que ha costado ese operador, sin contar sus hijos</li><li style=\'margin:10px 0;list-style-position:outside\'>El coste está donde <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>dbHits</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>rows</code> son altos; el del plan es la suma de todos los operadores</li><li style=\'margin:10px 0;list-style-position:outside\'>Dos planes se comparan por sus <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>dbHits</code> totales, no por cómo se ve el árbol</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Si el índice no se usa, la hoja es un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>NodeByLabelScan</code> y los <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>dbHits</code> lo delatan</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

Y con `PROFILE`, que además ejecuta la consulta y mide el trabajo real. `summary.profile` tiene la misma estructura de árbol que el plan de `EXPLAIN` (`operatorType`, `args`, `children`) y añade a cada operador dos medidas:

- **`rows`.** Filas que ha producido de verdad ese operador. Compáralas con `EstimatedRows` en `args`. Si de un operador al siguiente el número crece mucho, un recorrido está multiplicando filas; si cae mucho, un filtro llega tarde y antes se ha trabajado de más.
- **`dbHits`.** Veces que el operador ha pedido algo al almacenamiento: localizar un nodo, leer una propiedad, seguir una relación. Cuenta sólo ese operador, sin sus hijos; el coste de todo el plan es la suma de los `dbHits` de todos los operadores (lo hace la función `db_hits` del ejemplo «El índice se comprueba en el plan»). Busca los operadores donde `dbHits` y `rows` son altos a la vez.

In [ ]:
query = """
PROFILE
MATCH (u:User)-[:WROTE]->(q:Question)
RETURN u.Id AS user_id, count(q) AS num_questions
ORDER BY num_questions DESC
LIMIT 10
"""

result: EagerResult = driver.execute_query(query)
print("\nPerfil de ejecución para la consulta de usuarios con más preguntas:")
pp(result.summary.profile)

Tags más populares

In [ ]:
query = """
MATCH (t:Tag)-[:TAGS]->(q:Question)
RETURN t.TagName AS tag, count(q) AS num_questions
ORDER BY num_questions DESC
LIMIT 10
"""

result: EagerResult = driver.execute_query(query)
print("\nTop 10 tags más utilizados:")
for i, r in enumerate(result.records, 1):
    print(f"  {i}. #{r['tag']}: {r['num_questions']} preguntas")

Un predicado puede usar una expresión regular con `=~`. Preguntas de las etiquetas `java` o `c++`:

In [ ]:
query = """
MATCH p=(t:Tag)-[:TAGS]->(:Question) WHERE t.TagName =~ "^java$|^c[+][+]$" RETURN count(p);
"""
result: EagerResult = driver.execute_query(query)
for r in result.records:
    pp(r)

La siguiente consulta muestra los usuarios que preguntan por cada Tag. Sin límite imprimiría miles de líneas, así que se queda con 10 tags y 5 usuarios de cada uno (`LIMIT` y el corte de lista `[..5]`):

In [ ]:
query = """
MATCH (t:Tag)-->(:Question)<--(u:User)
RETURN t.TagName, collect(distinct u.Id)[..5] AS first_users
ORDER BY t.TagName
LIMIT 10
"""

result: EagerResult = driver.execute_query(query)
for r in result.records:
    pp(r)

El mismo `MATCH` se puede usar para encontrar qué conjunto de tags ha usado cada usuario cambiando lo que retornamos (con el mismo límite: 10 usuarios y 5 tags de cada uno):

In [ ]:
query = """
MATCH (t:Tag)-->(:Question)<--(u:User)
RETURN u.Id, collect(distinct t.TagName)[..5] AS first_tags
ORDER BY u.Id
LIMIT 10
"""

result: EagerResult = driver.execute_query(query)
for r in result.records:
    pp(r)

#### Numerar un ranking

Cypher no tiene una función que numere las filas de un resultado ordenado. El giro habitual es ordenar, recoger las filas en una lista con `collect()` y recorrer sus posiciones con `UNWIND range(...)`: la posición en la lista es el puesto. `ORDER BY` lleva un desempate para que el orden sea siempre el mismo.

In [ ]:
query = """
MATCH (t:Tag)-[:TAGS]->(q:Question)
WITH t, count(q) AS num_questions
ORDER BY num_questions DESC, t.TagName ASC
LIMIT 5
WITH collect({tag: t.TagName, num_questions: num_questions}) AS top
UNWIND range(0, size(top) - 1) AS idx
RETURN idx + 1 AS position, top[idx].tag AS tag, top[idx].num_questions AS num_questions
"""

result: EagerResult = driver.execute_query(query)
for r in result.records:
    print(f"  {r['position']}. #{r['tag']}: {r['num_questions']} preguntas")

#### Contar con una condición: `CASE`

Cypher tiene la misma expresión condicional que SQL: `CASE WHEN condición THEN valor ELSE otro_valor END`, con tantas ramas `WHEN` como haga falta. Es una expresión como cualquier otra, así que puede ir en `RETURN`, en `WITH`, en un `WHERE` o dentro de una agregación. Dentro de `sum()` sirve para contar sólo las filas que cumplen una condición: cada fila suma 1 si la cumple y 0 si no, igual que `SUM(CASE ...)` en SQL. Si se omite `ELSE`, el valor es `null`, y como `count()` ignora los nulos, `count(CASE WHEN ... THEN 1 END)` cuenta lo mismo.

La consulta siguiente calcula, para las cinco etiquetas con más preguntas, cuántas de ellas tienen una respuesta aceptada (una pregunta sin aceptada no tiene la propiedad `AcceptedAnswerId`, así que vale `null`) y qué porcentaje suponen. Como los dos recuentos son enteros, `/` hace la división entera y el porcentaje sale truncado. La segunda clasifica con varias ramas las preguntas por su puntuación.

In [ ]:
query = """
MATCH (t:Tag)-[:TAGS]->(q:Question)
WITH t, count(q) AS num_questions,
     sum(CASE WHEN q.AcceptedAnswerId IS NULL THEN 0 ELSE 1 END) AS with_accepted
ORDER BY num_questions DESC, t.TagName ASC
LIMIT 5
RETURN t.TagName AS tag, num_questions, with_accepted,
       100 * with_accepted / num_questions AS pct_accepted
"""

result: EagerResult = driver.execute_query(query)
print("Preguntas con respuesta aceptada en las 5 etiquetas más usadas:")
for r in result.records:
    print(f"  #{r['tag']}: {r['with_accepted']} de {r['num_questions']} ({r['pct_accepted']} %)")

query = """
MATCH (q:Question)
WITH CASE
       WHEN q.Score >= 10 THEN 'high'
       WHEN q.Score > 0 THEN 'positive'
       ELSE 'zero_or_negative'
     END AS score_band
RETURN score_band, count(*) AS num_questions
ORDER BY num_questions DESC
"""

result = driver.execute_query(query)
print("\nPreguntas según su puntuación:")
for r in result.records:
    print(f"  {r['score_band']}: {r['num_questions']}")

#### El índice se comprueba en el plan

La restricción de unicidad de `TagName` creó un índice. El primer perfil puede buscar `java` directamente; el segundo transforma cada valor con `toLower()` y dificulta ese acceso. Compara el `operatorType` de la hoja (`NodeIndexSeek` frente a `NodeByLabelScan`) y los `dbHits`, como se explicó con el primer `PROFILE`, no sólo que ambas consultas devuelvan el mismo nodo.

Como los `dbHits` de un operador no incluyen los de sus hijos, el total de un plan se obtiene recorriendo el árbol. Lo hace la función `db_hits` de la celda siguiente: cada operador es un diccionario con su `dbHits` y una lista `children`, y la recursión termina en las hojas.

In [ ]:
def db_hits(plan: dict) -> int:
    """Suma los *db hits* de un operador y de todos sus hijos."""
    propios: int = plan.get("dbHits", 0)
    de_los_hijos: int = sum(db_hits(hijo) for hijo in plan.get("children", []))
    return propios + de_los_hijos


indexed_profile: EagerResult = driver.execute_query(
    "PROFILE MATCH (t:Tag {TagName: 'java'}) RETURN t.TagName"
)
transformed_profile: EagerResult = driver.execute_query(
    "PROFILE MATCH (t:Tag) WHERE toLower(t.TagName) = 'java' RETURN t.TagName"
)
print("Predicado directo:")
pp(indexed_profile.summary.profile)
print("Predicado que transforma la propiedad:")
pp(transformed_profile.summary.profile)
print(
    f"db hits totales: {db_hits(indexed_profile.summary.profile)} (directo) frente a "
    f"{db_hits(transformed_profile.summary.profile)} (transformando)"
)

### EJERCICIO 3: comprobar el índice con `PROFILE`

La consulta de las etiquetas más populares recorre `Tag` sin filtrar por `TagName`, así que no usa el índice de la restricción de unicidad. Como en el ejemplo anterior, escribe una consulta que cuente las preguntas del tag `'python'` filtrando por `TagName` y compara su `PROFILE` con el de una versión que filtre con `toLower(t.TagName) = 'python'`. Guarda el resultado de cada consulta en `resultado_3_directo` y `resultado_3_transformado`.

Pistas:

- Las dos consultas cuentan las preguntas del mismo tag con `(t:Tag)-[:TAGS]->(q:Question)` y `count(q)`. Una fija `TagName` dentro del patrón, `(t:Tag {TagName: 'python'})`; la otra lo transforma en un `WHERE`. Empiezan con la palabra `PROFILE` y cada una devuelve una sola fila con una sola columna, el recuento.
- El plan se lee de `.summary.profile`, como en el ejemplo anterior. Imprímelo con `pp()` y fíjate en el operador de la hoja: debería ser `NodeUniqueIndexSeek` en la consulta directa y `NodeByLabelScan` (seguido de un `Filter`) en la transformada.
- El total de *db hits* de cada plan lo da la función `db_hits` del ejemplo anterior: `db_hits(resultado_3_directo.summary.profile)`.
- La comprobación espera que el predicado directo cueste como mucho lo que cuesta el transformado, y que el transformado no cueste cero.

In [ ]:
# Comprobación del EJERCICIO 3: comprobar el índice con PROFILE
assert "resultado_3_directo" in globals(), "Guarda la consulta directa en resultado_3_directo"
assert "resultado_3_transformado" in globals(), "Guarda la otra en resultado_3_transformado"
assert resultado_3_directo.summary.profile and resultado_3_transformado.summary.profile, (
    "Las dos consultas deben empezar por PROFILE"
)

esperado: int = driver.execute_query(
    "MATCH (t:Tag {TagName: 'python'})-[:TAGS]->(q:Question) RETURN count(q)"
).records[0][0]
assert resultado_3_directo.records[0][0] == esperado, (
    "El recuento de la consulta directa no coincide"
)
assert resultado_3_transformado.records[0][0] == esperado, (
    "El recuento de la consulta transformada no coincide"
)

hits_directo: int = db_hits(resultado_3_directo.summary.profile)
hits_transformado: int = db_hits(resultado_3_transformado.summary.profile)
print(f"db hits: {hits_directo} (directo) | {hits_transformado} (transformando)")
assert hits_transformado > 0, "La consulta transformada no ha leído nada"
assert hits_directo <= hits_transformado, (
    "Filtrar por TagName debe costar como mucho lo que cuesta transformar la propiedad"
)
print("EJERCICIO 3: comprobado")

La consulta de reciprocidad entre usuarios de Stackoverflow se puede resolver de manera muy fácil. En esta primera consulta se devuelven los nodos:

In [ ]:
query = """// Consulta de reciprocidad entre usuarios
MATCH
  (u1:User)-[:WROTE]->()-[:ANSWERS]->()<-[:WROTE]-(u2:User),
  (u2)-[:WROTE]->()-[:ANSWERS]->()<-[:WROTE]-(u1)
WHERE u1.Id < u2.Id
RETURN DISTINCT u1,u2
ORDER BY u1.Id, u2.Id
"""

result: EagerResult = driver.execute_query(query)
for r in result.records:
    print(r["u1"], r["u2"])

O bien retornar los `Id` de cada usuario:

In [ ]:
query = """
MATCH
  (u1:User)-[:WROTE]->()-[:ANSWERS]->()<-[:WROTE]-(u2:User),
  (u2)-[:WROTE]->()-[:ANSWERS]->()<-[:WROTE]-(u1)
WHERE u1.Id < u2.Id
RETURN DISTINCT u1.Id,u2.Id
ORDER BY u1.Id, u2.Id
"""

result: EagerResult = driver.execute_query(query)
for r in result.records:
    pp(r)

Y finalmente, la creación de relaciones `:RECIPROCATE` entre los usuarios, con la cláusula `WITH`.

`WITH` encadena dos partes de una consulta: a la siguiente sólo pasa lo que `WITH` nombra (con un alias `AS` si hace falta), y es donde se agrega, se ordena o se filtra un resultado intermedio. En este caso se usa `WITH` para demostración aunque no es estrictamente necesario. La consulta siguiente es la de reciprocidad de arriba, pero crea una relación `:RECIPROCATE` en cada sentido entre cada dos usuarios que se ayudan mutuamente.

In [ ]:
query = """
// Consulta de reciprocidad, creando relaciones RECIPROCATE
MATCH
  (u1:User)-[:WROTE]->()-[:ANSWERS]->()<-[:WROTE]-(u2:User),
  (u2)-[:WROTE]->()-[:ANSWERS]->()<-[:WROTE]-(u1)
WHERE u1.Id < u2.Id
WITH u1 AS user1, u2 AS user2
  MERGE (user1)-[:RECIPROCATE]->(user2)
  MERGE (user2)-[:RECIPROCATE]->(user1)
"""

result: EagerResult = driver.execute_query(query)
pp(result.summary.counters)

### Una relación derivada: intereses del usuario

La relación `INTERESTED_IN` materializa un recorrido frecuente: un usuario está interesado en las etiquetas sobre las que pregunta. `MERGE` hace idempotente la derivación. Es una optimización con coste de mantenimiento, no un hecho independiente de los datos de origen.

In [ ]:
result: EagerResult = driver.execute_query("""
MATCH (u:User)-[:WROTE]->(q:Question)-[:TAGGED_WITH]->(t:Tag)
MERGE (u)-[:INTERESTED_IN]->(t)
""")
pp(result.summary.counters)

In [ ]:
# @title Diapositiva: MATCH, relaciones y agregaciones
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">MATCH, relaciones y agregaciones</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Un patrón puede recorrer una o varias relaciones</li><li style=\'margin:10px 0;list-style-position:outside\'>Las agregaciones resumen el grafo sin convertirlo en tablas de hechos</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EXPLAIN</code> muestra el plan; <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>PROFILE</code> mide la ejecución</li><li style=\'margin:10px 0;list-style-position:outside\'>La cardinalidad y el índice influyen en el coste</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Ya sabemos leer el grafo; ahora buscaremos recorridos de varias etapas</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Para el examen: Cypher
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #8a6a00;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#8a6a00" size="8" noshade style="display:none"><div style="color:#8a6a00;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#8a6a00" style="color:inherit"><b>Para el examen</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Para el examen: Cypher</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MATCH</code> describe un patrón, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>WHERE</code> lo filtra y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>RETURN</code> proyecta</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>WITH</code> encadena etapas y es donde se agrega un resultado intermedio</li><li style=\'margin:10px 0;list-style-position:outside\'>La agrupación es implícita: agrupa todo lo que no sea función de agregación</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>OPTIONAL MATCH</code> conserva la fila aunque el patrón no case; un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>WHERE</code> mal colocado la borra</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>count(DISTINCT x)</code> frente a <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>count(x)</code>: los caminos repetidos inflan la cuenta</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #8a6a00;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pregunta típica: escribir el patrón que responde a una pregunta en lenguaje natural</div></blockquote><hr color="#8a6a00" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Recorridos más complejos
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Recorridos más complejos</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Seguir respuestas, reciprocidad e intereses hasta varios saltos</li><li style=\'margin:10px 0;list-style-position:outside\'>Usar <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>OPTIONAL MATCH</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>NOT EXISTS</code> sin perder nodos</li><li style=\'margin:10px 0;list-style-position:outside\'>Comparar caminos mínimos y relaciones derivadas</li><li style=\'margin:10px 0;list-style-position:outside\'>Limitar la expansión para que el recorrido siga siendo explicable</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Un grafo permite expresar la relación; el diseño decide cuánto recorrer</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### 4. Patrones de Grafo Más Complejos

Ahora exploremos patrones que involucran múltiples relaciones y nodos.

Encontrar usuarios que han respondido preguntas

In [ ]:
query = """
MATCH (user:User)-[:WROTE]->(answer:Answer)-[:ANSWERS]->(question:Question)
RETURN user.Id AS answerer, count(DISTINCT question) AS answered_questions
ORDER BY answered_questions DESC
LIMIT 10
"""

result: EagerResult = driver.execute_query(query)
print("\nTop 10 usuarios que más ayudan (responden preguntas):")
for i, r in enumerate(result.records, 1):
    print(f"  {i}. Usuario {r['answerer']}: {r['answered_questions']} preguntas respondidas")

Encontrar preguntas sin respuestas

In [ ]:
query = """
MATCH (q:Question)
WHERE NOT EXISTS { MATCH (q)-[:ANSWERS]-(:Answer) }
RETURN q.Id AS question_id, q.Title AS title, q.Score AS score
ORDER BY q.Score DESC
LIMIT 10
"""

result: EagerResult = driver.execute_query(query)
print("\nPreguntas sin respuestas (con mejor score):")
for r in result.records:
    print(f"  [{r['question_id']}] {r['title']} (Score: {r['score']})")

### `OPTIONAL MATCH`: conservar preguntas sin respuestas positivas

Un `MATCH` sólo devuelve las filas cuyo patrón encuentra; `OPTIONAL MATCH` conserva también la fila cuando no lo encuentra y deja a `null` las variables que no ha podido completar. Vamos a contar respuestas de puntuación positiva, incluyendo las preguntas que no tienen ninguna.

Para que la salida sea pequeña y se vean ambos casos, elegimos hasta tres preguntas con alguna respuesta positiva y hasta tres sin ninguna respuesta. La muestra contiene identificadores del grafo ya cargado; no se crean nodos ni relaciones.

In [ ]:
con_positivas: EagerResult = driver.execute_query("""
MATCH (q:Question)
WHERE EXISTS { MATCH (q)<-[:ANSWERS]-(a:Answer) WHERE a.Score > 0 }
RETURN q.Id AS id
LIMIT 3
""")
sin_respuestas: EagerResult = driver.execute_query("""
MATCH (q:Question)
WHERE NOT EXISTS { MATCH (q)<-[:ANSWERS]-(:Answer) }
RETURN q.Id AS id
LIMIT 3
""")
preguntas_muestra: list[int] = [row["id"] for row in con_positivas.records + sin_respuestas.records]
print(f"Preguntas de la muestra: {preguntas_muestra}")

El `WHERE` situado justo después de `OPTIONAL MATCH` forma parte del patrón opcional: busca sólo respuestas positivas. Si no encuentra ninguna, conserva la pregunta con `a` a `null`. Como `count(a)` no cuenta valores `null`, el resultado es cero.

In [ ]:
resultado_opcional: EagerResult = driver.execute_query(
    """
MATCH (q:Question)
WHERE q.Id IN $question_ids
OPTIONAL MATCH (q)<-[:ANSWERS]-(a:Answer)
WHERE a.Score > 0
RETURN q.Id AS question, count(a) AS positive_answers
ORDER BY question
LIMIT 6
""",
    question_ids=preguntas_muestra,
)
for fila in resultado_opcional.records:
    print(dict(fila))

Ahora pasamos `q` y `a` por `WITH` y aplicamos la misma condición después. Ese `WHERE` filtra las filas que ya ha producido `OPTIONAL MATCH`: cuando `a` es `null`, `a.Score > 0` no es verdadero y la fila desaparece. Por eso las preguntas sin respuestas ya no aparecen, en vez de salir con un cero.

In [ ]:
resultado_filtrado: EagerResult = driver.execute_query(
    """
MATCH (q:Question)
WHERE q.Id IN $question_ids
OPTIONAL MATCH (q)<-[:ANSWERS]-(a:Answer)
WITH q, a
WHERE a.Score > 0
RETURN q.Id AS question, count(a) AS positive_answers
ORDER BY question
LIMIT 6
""",
    question_ids=preguntas_muestra,
)
for fila in resultado_filtrado.records:
    print(dict(fila))

También se puede buscar un camino mínimo entre dos usuarios. El patrón `[*]` admite cualquier tipo de relación y la recorre en ambos sentidos: además de preguntas y respuestas, puede pasar por etiquetas o por las relaciones derivadas `:RECIPROCATE` e `:INTERESTED_IN`. Por tanto, la distancia depende también de las relaciones que se han creado previamente. Por ejemplo:

In [ ]:
query = "MATCH p=shortestPath( (u1:User {Id: 24})-[*]-(u2:User {Id:25}) ) RETURN p"

result: EagerResult = driver.execute_query(query)
for r in result.records:
    print(r["p"])

Con otra pareja de usuarios:

In [ ]:
query = "MATCH p=shortestPath( (u1:User {Id: 324})-[*]-(u2:User {Id:25}) ) RETURN p"

result: EagerResult = driver.execute_query(query)
for r in result.records:
    print(r["p"])

`allShortestPaths` devuelve todos los caminos que tienen la longitud mínima entre los dos usuarios. Como también admite cualquier tipo de relación, un camino puede utilizar directamente una relación derivada `:RECIPROCATE`, si existe:

In [ ]:
query = "MATCH p=allShortestPaths( (u1:User {Id: 24})-[*]-(u2:User {Id:25}) ) RETURN p"

result: EagerResult = driver.execute_query(query)
for r in result.records:
    print(r["p"])

**Ejemplo**: usuarios conectados por un camino de uno a tres saltos de `:RECIPROCATE` que comparten al menos dos etiquetas en sus preguntas. Cada salto es una relación derivada entre dos usuarios que se han respondido mutuamente; el patrón se recorre en ambos sentidos. Se muestran los primeros 20 resultados, ordenados por número de etiquetas compartidas y longitud del camino. Una misma pareja puede aparecer varias veces si hay varios caminos.

In [ ]:
query = """
MATCH path=(u1:User)-[:RECIPROCATE*1..3]-(u2:User)
WHERE u1.Id < u2.Id
WITH u1, u2, path
MATCH (u1)-[:INTERESTED_IN]->(t:Tag)<-[:INTERESTED_IN]-(u2)
WITH u1, u2, collect(DISTINCT t.TagName) AS common_tags, length(path) AS distance
WHERE size(common_tags) >= 2
RETURN u1.Id AS user1, u2.Id AS user2, common_tags, distance
ORDER BY size(common_tags) DESC, distance ASC
LIMIT 20
"""

result: EagerResult = driver.execute_query(query)
print("Pares de usuarios conectados con intereses comunes:")
for r in result.records:
    print(
        f"Usuario {r['user1']} <-> Usuario {r['user2']}: {len(r['common_tags'])} tags en común - Distancia: {r['distance']}"
    )
    print(f"  Tags: {', '.join(r['common_tags'][:5])}")  # Mostrar solo primeros 5 tags

In [ ]:
# @title Diapositiva: Del patrón al coste
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Del patrón al coste</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Las consultas expresan recorridos y agregaciones sobre nodos relacionados</li><li style=\'margin:10px 0;list-style-position:outside\'>Las relaciones derivadas pueden abaratar preguntas frecuentes</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EXPLAIN</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>PROFILE</code> conectan la consulta con el trabajo del motor</li><li style=\'margin:10px 0;list-style-position:outside\'>Falta comprobar qué sucede si una carga se repite</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La identidad y la idempotencia son tan importantes como el resultado</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Para el examen: recorridos y coste
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #8a6a00;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#8a6a00" size="8" noshade style="display:none"><div style="color:#8a6a00;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#8a6a00" style="color:inherit"><b>Para el examen</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Para el examen: recorridos y coste</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Un recorrido de varios saltos multiplica caminos: hay que acotar la profundidad</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>shortestPath</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>allShortestPaths</code> no son lo mismo que recorrer y ordenar</li><li style=\'margin:10px 0;list-style-position:outside\'>Una relación derivada abarata la lectura y añade coste de mantenimiento</li><li style=\'margin:10px 0;list-style-position:outside\'>El grafo gana cuando la pregunta es de conexión, no de agregación masiva</li><li style=\'margin:10px 0;list-style-position:outside\'>La dirección de la relación cambia el resultado y también el plan</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #8a6a00;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pregunta típica: decir cuándo compensa materializar una relación derivada</div></blockquote><hr color="#8a6a00" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: CREATE frente a MERGE
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px"><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>CREATE</code> frente a <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code></h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Ejecutar primero una carga deliberadamente duplicadora</li><li style=\'margin:10px 0;list-style-position:outside\'>Contar los nodos creados y limpiar sólo la demostración</li><li style=\'margin:10px 0;list-style-position:outside\'>Repetir con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> sobre la identidad correcta</li><li style=\'margin:10px 0;list-style-position:outside\'>Aplicar propiedades con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>SET</code> y comprobar que la carga es idempotente</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Una carga repetible es una propiedad del modelo, no sólo de la herramienta</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## `CREATE` frente a `MERGE`: ejecutar primero la versión mala

Esta demostración aislada crea dos nodos con la misma identidad lógica y cuenta el duplicado. Después limpia únicamente la etiqueta temporal y repite la operación con `MERGE`. Conservar ambos resultados hace visible un problema que se perdería si sólo enseñásemos la solución final.

In [ ]:
driver.execute_query("MATCH (n:IdempotencyDemo) DETACH DELETE n")
driver.execute_query("UNWIND range(1, 2) AS attempt CREATE (:IdempotencyDemo {Id: 1})")
bad_result: EagerResult = driver.execute_query(
    "MATCH (n:IdempotencyDemo {Id: 1}) RETURN count(n) AS copies"
)
print("Después de CREATE:", bad_result.records[0]["copies"], "copias")

driver.execute_query("MATCH (n:IdempotencyDemo) DETACH DELETE n")
driver.execute_query("UNWIND range(1, 2) AS attempt MERGE (:IdempotencyDemo {Id: 1})")
good_result: EagerResult = driver.execute_query(
    "MATCH (n:IdempotencyDemo {Id: 1}) RETURN count(n) AS copies"
)
print("Después de MERGE:", good_result.records[0]["copies"], "copia")

### `MERGE` empareja el patrón completo

`MERGE` busca el patrón **entero**, con todas sus propiedades, y sólo lo crea si no lo encuentra. Si en el patrón va una propiedad que cambia de una carga a otra, deja de encontrar el nodo y crea otro: la carga vuelve a duplicar aunque use `MERGE`. Por eso en el patrón va sólo la identidad y lo demás se escribe después con `SET`. `ON CREATE SET` y `ON MATCH SET` separan lo que se hace al dar de alta el nodo de lo que se hace al encontrarlo.

In [ ]:
# Dos cargas del mismo nodo; `visits` cambia de una a otra.
driver.execute_query("MATCH (n:IdempotencyDemo) DETACH DELETE n")
for visits in (10, 11):
    driver.execute_query("MERGE (:IdempotencyDemo {Id: 1, visits: $visits})", visits=visits)
full_pattern: EagerResult = driver.execute_query(
    "MATCH (n:IdempotencyDemo {Id: 1}) RETURN count(n) AS copies"
)
print("MERGE con visits en el patrón:", full_pattern.records[0]["copies"], "copias")

driver.execute_query("MATCH (n:IdempotencyDemo) DETACH DELETE n")
for visits in (10, 11):
    driver.execute_query(
        """
        MERGE (n:IdempotencyDemo {Id: 1})
        ON CREATE SET n.loads = 1
        ON MATCH SET n.loads = n.loads + 1
        SET n.visits = $visits
        """,
        visits=visits,
    )
identity: EagerResult = driver.execute_query(
    "MATCH (n:IdempotencyDemo {Id: 1}) "
    "RETURN count(n) AS copies, n.visits AS visits, n.loads AS loads"
)
print("MERGE sólo con la identidad:", dict(identity.records[0]))
driver.execute_query("MATCH (n:IdempotencyDemo) DETACH DELETE n")

In [ ]:
# @title Diapositiva: De la consulta al modelo derivado
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">De la consulta al modelo derivado</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Hemos creado identidades, nodos, relaciones y etiquetas</li><li style=\'margin:10px 0;list-style-position:outside\'>Los recorridos y agregaciones muestran qué preguntas hace barato el grafo</li><li style=\'margin:10px 0;list-style-position:outside\'>Las relaciones derivadas y los índices tienen un coste de mantenimiento</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> permite repetir una carga si la identidad está bien elegida</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Los ejercicios evaluables combinan carga, relaciones derivadas y recorridos</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Para el examen: índices y planes
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #8a6a00;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#8a6a00" size="8" noshade style="display:none"><div style="color:#8a6a00;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#8a6a00" style="color:inherit"><b>Para el examen</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Para el examen: índices y planes</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Una restricción de unicidad crea el índice que <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> necesita para no degradarse</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EXPLAIN</code> muestra el plan estimado; <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>PROFILE</code> lo ejecuta y cuenta <em>db hits</em></li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>NodeIndexSeek</code> frente a <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>NodeByLabelScan</code>: ahí se ve si el índice sirve</li><li style=\'margin:10px 0;list-style-position:outside\'>Transformar la propiedad en el <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>WHERE</code> (<code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>toLower(...)</code>) impide usar el índice</li><li style=\'margin:10px 0;list-style-position:outside\'>Sin índice, un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> masivo degrada a recorrido por etiqueta</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #8a6a00;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pregunta típica: leer un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>PROFILE</code> y decir dónde se concentra el coste</div></blockquote><hr color="#8a6a00" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: El boletín evaluable
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">El boletín evaluable</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Los ejercicios 4 al 10 son los que hay que hacer para la entrega</li><li style=\'margin:10px 0;list-style-position:outside\'>Los parámetros con los que trabajas salen de tu identificador</li><li style=\'margin:10px 0;list-style-position:outside\'>El resultado final es pequeño, ordenado y se lee de tu propio nodo de boletín</li><li style=\'margin:10px 0;list-style-position:outside\'>Se entrega código y resultado</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Hacer los ejercicios en orden deja preparadas las estructuras del resultado final</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

# Ejercicios evaluables

El boletín contiene una serie de **ejercicios evaluables**.

De cada ejercicio se requiere: **(a)** código y resultado; y **(b)** cuando se solicite, una medida o un plan interpretado.

Cuando un enunciado pida explicar o interpretar algo, escríbelo como comentario en la propia celda de la solución, antes o después del código. No se envía con la entrega, pero es lo que tendrás que defender sin el cuaderno.

## Preparación de la entrega

Los ejercicios evaluables y la entrega dependen del identificador que escribiste al principio del cuaderno, en «Identificación de la entrega». La celda siguiente te lo vuelve a mostrar: **compruébalo de nuevo** antes de seguir. De él salen tus parámetros y la huella con la que se registra el resultado, así que un identificador equivocado obliga a repetir los ejercicios. Si no es el tuyo, corrígelo en aquella celda, vuelve a ejecutarla y continúa desde aquí.

In [ ]:
print(f"El DNI que has seleccionado para la entrega es: {DNI}")
print("Compruébalo de nuevo antes de continuar.")

Las dos celdas siguientes preparan lo que usa el resto del boletín. La primera descarga el cliente de entregas del repositorio público `dsevilla/bdge`, así que no hay que copiar ningún ZIP junto al cuaderno. La segunda calcula con él tu huella. Ninguna de las dos envía nada: el envío es el último paso, cuando hayas terminado los ejercicios.

In [ ]:
import importlib.util
import json
import pathlib
import re
import sys
import urllib.error
import urllib.request
import zipfile
from types import ModuleType

CLIENTE_BASE_URL: str = (
    "https://raw.githubusercontent.com/dsevilla/bdge/26-27/addendum/practice-evidence/dist"
)


def carga_cliente_evidencias() -> ModuleType:
    """Consulta la versión publicada, descarga el cliente para este intérprete y lo importa."""
    if "bdge_evidence" in sys.modules:
        return sys.modules["bdge_evidence"]

    cache_tag: str = sys.implementation.cache_tag
    manifiesto_url: str = f"{CLIENTE_BASE_URL}/manifest.json"
    try:
        with urllib.request.urlopen(manifiesto_url, timeout=30) as respuesta:
            manifiesto = json.loads(respuesta.read().decode("utf-8"))
    except (urllib.error.URLError, TimeoutError, ValueError) as error:
        raise RuntimeError(
            f"No se pudo consultar la versión del cliente en {manifiesto_url}"
        ) from error

    if not isinstance(manifiesto, dict) or manifiesto.get("format") != "bdge-evidence-release/v1":
        raise RuntimeError(
            f"El manifiesto del cliente no tiene un formato reconocido: {manifiesto_url}"
        )
    version_cliente = manifiesto.get("client_release")
    if not isinstance(version_cliente, str) or not re.fullmatch(
        r"[0-9]+\.[0-9]+\.[0-9]+", version_cliente
    ):
        raise RuntimeError(f"El manifiesto no contiene una versión válida: {manifiesto_url}")

    nombre: str = f"bdge-evidence-{version_cliente}-{cache_tag}.zip"
    destino: pathlib.Path = pathlib.Path.cwd() / nombre
    if not destino.exists():
        url: str = f"{CLIENTE_BASE_URL}/{nombre}"
        try:
            with urllib.request.urlopen(url, timeout=30) as respuesta:
                destino.write_bytes(respuesta.read())
        except urllib.error.HTTPError as error:
            raise RuntimeError(
                f"No hay cliente publicado para {cache_tag}: {url} ({error.code})"
            ) from error
        print(f"Cliente descargado: {nombre}")

    # El ZIP lleva bytecode y manifiesto, pero debe coincidir con este Python.
    with zipfile.ZipFile(destino) as archivo:
        magico: bytes = archivo.read("bdge_evidence.pyc")[:4]
        manifiesto_zip = json.loads(archivo.read("manifest.json").decode("utf-8"))
    if magico != importlib.util.MAGIC_NUMBER:
        raise RuntimeError(
            f"{nombre} se compiló con otro Python y no puede cargarse en este "
            f"kernel ({sys.version.split()[0]}, {cache_tag}). Borra el fichero "
            "y vuelve a ejecutar esta celda."
        )
    if (
        not isinstance(manifiesto_zip, dict)
        or manifiesto_zip.get("client_version") != f"bdge-evidence-demo/{version_cliente}"
    ):
        raise RuntimeError(f"El ZIP no corresponde a la versión publicada: {nombre}")

    sys.path.insert(0, str(destino))
    import bdge_evidence

    return bdge_evidence


evidence: ModuleType = carga_cliente_evidencias()
print(f"Cliente de evidencias {evidence.CLIENT_VERSION} disponible")

In [ ]:
HUELLA: str = evidence.hash_student_identifier(DNI, CURSO, BOLETIN)

if evidence.is_example_identifier(DNI):
    print("AVISO: sigues usando el identificador de ejemplo. Cámbialo antes de entregar.")
print(f"Curso: {CURSO} | Boletín: {BOLETIN}")
print(f"Huella del alumno (esto es lo único que se registra): {HUELLA}")

## Parámetros personales del boletín

El DNI no selecciona un caso de una lista: sus dígitos entran directamente
en las consultas. Los rangos están acotados para que todos trabajéis con un
volumen razonable. La letra final no interviene ni en los cálculos ni en la
huella que identifica la entrega.

In [ ]:
from dataclasses import dataclass
from pprint import pprint


@dataclass(frozen=True)
class ParametrosBoletin:
    numero: int
    umbral_reputacion: int
    año_inicial: int
    mes: int
    minimo_actividad: int
    limite: int
    factor_a: int
    factor_b: int


NUMERO_DNI: int = int(evidence.normalize_student_identifier(DNI))
PARAMETROS: ParametrosBoletin = ParametrosBoletin(
    numero=NUMERO_DNI,
    umbral_reputacion=100 + NUMERO_DNI % 4_900,
    año_inicial=2016 + (NUMERO_DNI // 10) % 5,
    mes=1 + (NUMERO_DNI // 100) % 12,
    minimo_actividad=2 + (NUMERO_DNI // 1_000) % 4,
    limite=5,
    factor_a=2 + NUMERO_DNI % 5,
    factor_b=3 + (NUMERO_DNI // 10) % 7,
)
pprint(PARAMETROS)

### Tu año

Los ejercicios estudian **un año completo del foro**: el año natural `PARAMETROS.año_inicial`, del 1 de enero al 31 de diciembre. Con el año 2017, por ejemplo, va de enero a diciembre de 2017. La pregunta que contestan es **la ayuda de quién llega más lejos ese año**: un autor ayuda a otro cuando responde a una pregunta suya, y su ayuda llega más lejos si esa persona ayuda a su vez a una tercera. Se estudia sólo entre los *autores habituales*: usuarios con reputación alta que además escribieron a menudo (preguntas o respuestas). Es una pregunta de recorridos, y el grafo la contesta sin agregar nada a mano. Para llegar a la respuesta, los ejercicios, en orden:

- eligen a los autores habituales del año
- crean quién ayudó a quién, con las respuestas del año a preguntas del año
- encadenan dos ayudas: `a` ayudó a `b`, `b` ayudó a `c` y `a` no ayudó directamente a `c`
- resumen, para cada autor, a cuánta gente ayudó directamente y a cuánta llega de forma indirecta
- entregan los autores cuya ayuda llega más lejos

Cada ejercicio usa lo que han dejado los anteriores, por eso se hacen en orden. En todos, el año se filtra con `CreationDate.year = $year`, donde `$year` es `PARAMETROS.año_inicial`.

### EJERCICIO 4: autores elegibles

Los nodos `:User` del grafo sólo tienen `Id`, porque se crearon a partir de los posts: la reputación está en `Users.parquet`. Lee ese fichero por lotes y localiza con `UNWIND` los nodos `:User` cuya reputación (`Reputation`) alcanza `PARAMETROS.umbral_reputacion`; no hace falta guardar la reputación en el grafo. Exige además que hayan escrito al menos `PARAMETROS.minimo_actividad` publicaciones con `CreationDate` dentro de tu año. Aquí una *publicación* es una pregunta o una respuesta: un nodo `:Post` con `PostTypeId` 1 o 2 (`p.PostTypeId IN [1, 2]`). El resto de nodos `:Post` que hay en el grafo (por ejemplo, las wikis de etiqueta, con `PostTypeId` de 4 a 7) no son preguntas ni respuestas del foro y no cuentan. Deja como resultado:

- los nodos `:User` elegibles, con la etiqueta adicional `:B7Author`;
- un nodo `(:B7Bulletin {fingerprint})` con tu `HUELLA`, que además guarda `course` y `bulletin`;
- una relación `(:B7Bulletin)-[:B7_INCLUDES]->(:B7Author)` por cada autor elegible.

Pistas:

- `fsspec` y `pyarrow.parquet` ya están importados más arriba. El fichero se abre en remoto, sin descargarlo entero: `with fsspec.open(f"{PARQUET_BASE_URL}/Users.parquet", "rb") as stream:` y, dentro, `parquet = pq.ParquetFile(stream)`.
- `parquet.iter_batches(columns=["Id", "Reputation"], batch_size=10_000)` es un iterador: cada vuelta del `for` entrega un lote de hasta 10.000 filas, y en memoria sólo hay uno a la vez. Con `columns` sólo se leen las columnas que pides, porque Parquet guarda cada una por separado.
- Un lote (`RecordBatch`) no se recorre directamente: `batch.to_pylist()` lo convierte en una lista de diccionarios, uno por fila, con `row["Id"]` y `row["Reputation"]`. `Id` y `Reputation` pueden venir a `None`, así que comprueba antes de comparar.
- Filtra la reputación en Python y manda a Neo4j sólo los `Id` que pasan, como lista de mapas en un parámetro: `driver.execute_query("UNWIND $rows AS row MATCH (u:User {Id: row.Id}) ...", rows=rows)`. Si el lote se queda sin ninguna fila, no hace falta enviar nada.
- Una forma de ordenar el trabajo: marca esos usuarios con una etiqueta temporal, cuenta después sus publicaciones del año en una sola consulta (`count(p)`, `WITH ... WHERE posts >= $minimum`, con `p.CreationDate.year = $year`) y quita la etiqueta temporal al terminar. Los parámetros `$year`, `$minimum`, `$fingerprint`, `$course` y `$bulletin` se pasan como argumentos con nombre, igual que `question_id`.
- La celda se puede volver a ejecutar: borra antes las relaciones `:B7_INCLUDES` y la etiqueta `:B7Author` de la ejecución anterior, y piensa si el nodo del boletín y sus relaciones deben crearse con `CREATE` o con `MERGE`: qué pasaría con cada uno si la celda se ejecuta dos veces.

In [ ]:
# Comprobación del EJERCICIO 4: autores elegibles
incluidos: list[Record] = driver.execute_query(
    "MATCH (:B7Bulletin {fingerprint: $fingerprint})-[:B7_INCLUDES]->(u:B7Author) RETURN u.Id",
    fingerprint=HUELLA,
).records
autores: list[int] = [identificador for (identificador,) in incluidos]
assert autores, "No hay ningún :B7Author enlazado al boletín con :B7_INCLUDES"

# Camino paralelo, sobre una muestra de 15. Los nodos :User sólo tienen `Id` (se
# crearon con MERGE desde Posts): Reputation no está en el grafo, así que se
# contrasta con Users.parquet, leyendo sólo esas dos columnas y quedándose con los
# usuarios de la muestra. La actividad se cuenta con una consulta propia.
muestra: list[int] = autores[:15]
en_muestra: set[int] = set(muestra)
usuarios: dict[int, dict[str, object]] = {}
with fsspec.open(f"{PARQUET_BASE_URL}/Users.parquet", "rb") as stream:
    parquet: pq.ParquetFile = pq.ParquetFile(stream)
    for lote in parquet.iter_batches(columns=["Id", "Reputation"], batch_size=50_000):
        for fila in lote.to_pylist():
            if fila["Id"] in en_muestra:
                usuarios[fila["Id"]] = fila

for user_id in muestra:
    assert user_id in usuarios, f"Usuario {user_id}: no existe en Users.parquet"
    datos: dict[str, object] = usuarios[user_id]
    assert (
        datos["Reputation"] is not None and datos["Reputation"] >= PARAMETROS.umbral_reputacion
    ), f"Usuario {user_id}: no alcanza el umbral de reputación"
    publicaciones: int = driver.execute_query(
        "MATCH (:User {Id: $id})-[:WROTE]->(p:Post) "
        "WHERE p.PostTypeId IN [1, 2] AND p.CreationDate.year = $year "
        "RETURN count(p)",
        id=user_id,
        year=PARAMETROS.año_inicial,
    ).records[0][0]
    assert publicaciones >= PARAMETROS.minimo_actividad, (
        f"Usuario {user_id}: no llega a la actividad mínima en el año"
    )
print(
    f"Autores elegibles: {len(autores):,} | verificados {len(muestra)} contra Users.parquet y Post"
)
print("EJERCICIO 4: comprobado")

### EJERCICIO 5: quién ayudó a quién

Crea `:B7_REPLIED_TO` entre autores: de `a` a `b` cuando `a` respondió a alguna pregunta de `b`. Es una relación por pareja ordenada de `:B7Author`, que cuenta sólo respuestas y preguntas con `CreationDate` dentro de tu año y excluye las autorrespuestas (`a <> b`). Cada relación lleva:

- `answers`: cuántas respuestas distintas de `a` hay a preguntas de `b`
- `accepted`: cuántas de esas respuestas son la aceptada de su pregunta (`Question.AcceptedAnswerId` igual al `Id` de la respuesta)

Pista: `accepted` es un recuento con condición. Cypher admite `CASE WHEN … THEN … ELSE … END` como SQL, y dentro de `sum()` cuenta sólo las filas que la cumplen (véase el ejemplo «Contar con una condición: `CASE`»).

In [ ]:
# Comprobación del EJERCICIO 5: quién ayudó a quién
respondio: int = driver.execute_query(
    "MATCH (:B7Author)-[r:B7_REPLIED_TO]->(:B7Author) RETURN count(r)"
).records[0][0]
assert respondio > 0, "No se ha creado ninguna relación :B7_REPLIED_TO"
print(f":B7_REPLIED_TO creadas: {respondio:,}")

# Camino paralelo, sobre una muestra de 10: cada relación se recalcula con una
# consulta propia, sin pasar por los nodos intermedios de la solución.
muestra: list[Record] = driver.execute_query(
    "MATCH (a:B7Author)-[r:B7_REPLIED_TO]->(b:B7Author) "
    "RETURN a.Id AS source, b.Id AS target, r.answers AS answers, r.accepted AS accepted "
    "LIMIT 10"
).records
for fila in muestra:
    esperado: Record = driver.execute_query(
        "MATCH (a:User {Id: $source})-[:WROTE]->(r:Answer)-[:ANSWERS]->(q:Question)"
        "<-[:WROTE]-(b:User {Id: $target}) "
        "WHERE q.CreationDate.year = $year AND r.CreationDate.year = $year "
        "RETURN count(DISTINCT r) AS answers, "
        "count(DISTINCT CASE WHEN q.AcceptedAnswerId = r.Id THEN r END) AS accepted",
        source=fila["source"],
        target=fila["target"],
        year=PARAMETROS.año_inicial,
    ).records[0]
    assert esperado["answers"] == fila["answers"], (
        f"{fila['source']} -> {fila['target']}: answers no coincide"
    )
    assert esperado["accepted"] == fila["accepted"], (
        f"{fila['source']} -> {fila['target']}: accepted no coincide"
    )

autorespuestas: int = driver.execute_query(
    "MATCH (a:B7Author)-[:B7_REPLIED_TO]->(a) RETURN count(*)"
).records[0][0]
assert autorespuestas == 0, ":B7_REPLIED_TO no debe apuntar de un autor a sí mismo"
print(f"Verificadas {len(muestra)} relaciones :B7_REPLIED_TO contra Answer/Question")
print("EJERCICIO 5: comprobado")

### EJERCICIO 6: hasta dónde llega la ayuda

Las relaciones `:B7_REPLIED_TO` forman un grafo dirigido de ayudas entre tus autores. Toma como origen al autor que ayudó a más autores distintos (si hay empate, el de menor `Id`). Siguiendo `:B7_REPLIED_TO` siempre en su dirección, calcula con `shortestPath` a cuántos saltos está cada autor al que llega su ayuda, directa o indirectamente.

Devuelve una fila por autor alcanzado, sin contar el origen, con dos columnas: su `Id` y el número de saltos. Ordena de más a menos saltos y, a igualdad, de menor a mayor `Id`: así el número de filas dice a cuántos autores llega la ayuda, y la primera, quién es el más lejano. Guarda el resultado de la consulta en `resultado_6`.

Explica en un comentario qué ventaja y qué coste tiene calcular estas distancias con un recorrido de longitud variable cada vez que se necesitan, frente a guardarlas en el grafo como relaciones.

Pistas:

- `-[:B7_REPLIED_TO*]->` admite caminos de cualquier longitud, y la flecha fija la dirección.
- Para tener la distancia a cada autor, combina el origen con todos los `:B7Author` distintos de él y aplica `shortestPath` a cada pareja; `length(p)` es el número de saltos del camino. Si un autor no es alcanzable, `MATCH` no devuelve ninguna fila para él.
- El origen se puede elegir en la misma consulta: ordena y limita con `WITH ... ORDER BY ... LIMIT 1` antes de buscar los caminos.

In [ ]:
# Comprobación del EJERCICIO 6: hasta dónde llega la ayuda
from collections import defaultdict, deque

assert "resultado_6" in globals(), "Guarda lo que devuelve execute_query en resultado_6"
assert len(resultado_6.keys) == 2, "Devuelve dos columnas: Id del autor alcanzado y saltos"

# Camino paralelo: las aristas de :B7_REPLIED_TO, que ya se validaron en un ejercicio
# anterior, y una búsqueda en anchura en Python desde el origen.
aristas: list[Record] = driver.execute_query(
    "MATCH (a:B7Author)-[:B7_REPLIED_TO]->(b:B7Author) RETURN a.Id, b.Id"
).records
assert aristas, ":B7_REPLIED_TO está vacía: completa antes el ejercicio que la crea"
sucesores: dict[int, set[int]] = defaultdict(set)
for origen, destino in aristas:
    sucesores[origen].add(destino)

origen_esperado: int = min(sucesores, key=lambda autor: (-len(sucesores[autor]), autor))
distancia: dict[int, int] = {origen_esperado: 0}
pendientes: deque[int] = deque([origen_esperado])
while pendientes:
    actual: int = pendientes.popleft()
    for siguiente in sucesores[actual]:
        if siguiente not in distancia:
            distancia[siguiente] = distancia[actual] + 1
            pendientes.append(siguiente)
del distancia[origen_esperado]

esperado: list[tuple[int, int]] = sorted(distancia.items(), key=lambda par: (-par[1], par[0]))
obtenido: list[tuple[int, int]] = [tuple(fila) for fila in resultado_6.records]
print(f"Autores alcanzados: esperados {len(esperado)} | obtenidos {len(obtenido)}")
assert dict(obtenido) == distancia and len(obtenido) == len(esperado), (
    "Los autores o sus distancias no coinciden (el origen es el autor que ayudó a más autores "
    "distintos; si hay empate, el de menor Id)"
)
assert obtenido == esperado, "Ordena de más a menos saltos y, a igualdad, de menor a mayor Id"
print("EJERCICIO 6: comprobado")

### EJERCICIO 7: `OPTIONAL MATCH` y autores sin interacción

La consulta siguiente pretende contar tus autores y, de ellos, cuántos no han ayudado a nadie: los que no tienen ninguna `:B7_REPLIED_TO` de salida con alguna respuesta (`answers > 0`). Parte de las `:B7_REPLIED_TO` que ya has creado.

```cypher
MATCH (u:B7Author)
OPTIONAL MATCH (u)-[r:B7_REPLIED_TO]->()
WITH u, r
WHERE r.answers > 0
WITH u, count(r) AS replies
RETURN count(*) AS authors,
       sum(CASE WHEN replies = 0 THEN 1 ELSE 0 END) AS no_interaction
```

1. Ejecútala tal cual y guarda el resultado de la consulta en `resultado_7_defectuosa`. Compara su `authors` con el número total de `:B7Author` y con su `no_interaction`: algo no cuadra.
2. Explica en un comentario qué línea causa el problema, qué filas elimina y por qué.
3. Corrígela para que cuente a todos los autores sin cambiar lo que pretende contar. Guarda el resultado de la consulta corregida, con las mismas dos columnas y en el mismo orden, en `resultado_7_corregida`.
4. Ejecuta la consulta corregida con `PROFILE` e indica en un comentario qué operador del plan corresponde al `OPTIONAL MATCH` y cuántas filas salen de él.

Si no ves el fallo, devuelve `u.Id` y `r` justo después del `OPTIONAL MATCH` (con un `LIMIT`) y fíjate en qué vale `r` para un autor que no ha ayudado a nadie, y en qué da una comparación con ese valor.

In [ ]:
# Comprobación del EJERCICIO 7: OPTIONAL MATCH y autores sin interacción
assert "resultado_7_defectuosa" in globals(), (
    "Guarda el resultado de la consulta del enunciado en resultado_7_defectuosa"
)
assert "resultado_7_corregida" in globals(), (
    "Guarda el resultado de la consulta corregida en resultado_7_corregida"
)
autores_defectuosa: int
autores_corregida: int
autores_sin_interaccion: int
autores_defectuosa, _ = resultado_7_defectuosa.records[0]
autores_corregida, autores_sin_interaccion = resultado_7_corregida.records[0]

# Camino paralelo: el total de autores y, con NOT EXISTS en lugar de OPTIONAL MATCH,
# los que no tienen ninguna :B7_REPLIED_TO de salida con respuestas.
total_autores: int = driver.execute_query("MATCH (u:B7Author) RETURN count(u)").records[0][0]
sin_interaccion_esperados: int = driver.execute_query(
    "MATCH (u:B7Author) "
    "WHERE NOT EXISTS { MATCH (u)-[r:B7_REPLIED_TO]->() WHERE r.answers > 0 } "
    "RETURN count(u)"
).records[0][0]
print(
    f"Total de autores: {total_autores} | defectuosa: {autores_defectuosa} | corregida: {autores_corregida}"
)
print(
    f"Sin interacción: esperados {sin_interaccion_esperados} | obtenidos {autores_sin_interaccion}"
)
assert autores_defectuosa == total_autores - sin_interaccion_esperados, (
    "resultado_7_defectuosa debe ser lo que devuelve la consulta del enunciado tal cual: "
    "pierde justo a los autores sin interacción"
)
assert autores_corregida == total_autores, "La consulta corregida debe contar a todos los autores"
assert autores_sin_interaccion == sin_interaccion_esperados, (
    "El número de autores sin interacción no coincide"
)
print("EJERCICIO 7: comprobado")

### EJERCICIO 8: cadenas de dos ayudas

Desde `:B7_REPLIED_TO`, crea `:B7_CHAIN` de `a` a `c` por cada par de autores distintos tal que `a` ayudó a algún `b`, ese `b` ayudó a `c`, y `a` **no** ayudó directamente a `c`. Es una sola relación por pareja ordenada (`a`, `c`), aunque haya varios `b` intermedios, y lleva:

- `via`: cuántos autores `b` distintos enlazan a `a` con `c`

Demuestra que repetir la consulta no duplica relaciones.


In [ ]:
# Comprobación del EJERCICIO 8: cadenas de dos ayudas
from collections import defaultdict

respondio: list[Record] = driver.execute_query(
    "MATCH (a:B7Author)-[:B7_REPLIED_TO]->(b:B7Author) RETURN a.Id AS source, b.Id AS target"
).records
assert respondio, ":B7_REPLIED_TO está vacía: completa antes el ejercicio que la crea"

# Camino paralelo: recalculado en Python puro a partir de :B7_REPLIED_TO, que
# ya se validó en un ejercicio anterior.
ayuda_a: dict[int, set[int]] = defaultdict(set)
for fila in respondio:
    ayuda_a[fila["source"]].add(fila["target"])

esperado: dict[tuple[int, int], int] = {}
for origen, intermedios in ayuda_a.items():
    destinos: dict[int, int] = defaultdict(int)
    for intermedio in intermedios:
        for destino in ayuda_a.get(intermedio, set()):
            if destino != origen and destino not in intermedios:
                destinos[destino] += 1
    for destino, via in destinos.items():
        esperado[(origen, destino)] = via

cadenas: list[Record] = driver.execute_query(
    "MATCH (a:B7Author)-[r:B7_CHAIN]->(c:B7Author) "
    "RETURN a.Id AS source, c.Id AS target, r.via AS via"
).records
obtenido: dict[tuple[int, int], int] = {
    (fila["source"], fila["target"]): fila["via"] for fila in cadenas
}
print(f"Cadenas: esperadas {len(esperado):,} | obtenidas {len(obtenido):,}")
assert len(cadenas) == len(obtenido), "Hay más de una relación :B7_CHAIN para alguna pareja"
assert set(obtenido) == set(esperado), "Las parejas (origen, destino) no coinciden"
for clave, via in esperado.items():
    assert obtenido[clave] == via, f"{clave}: via no coincide"
print("EJERCICIO 8: comprobado")

### EJERCICIO 9: alcance de la ayuda

Resume, para cada autor que tenga alguna `:B7_CHAIN` de salida, hasta dónde llega su ayuda. Crea una relación `(:B7Bulletin {fingerprint})-[:B7_REACH]->(:B7Author)` por autor, con:

- `helped`: a cuántos autores distintos ayudó directamente (relaciones `:B7_REPLIED_TO` de salida)
- `reach`: a cuántos autores distintos llega de forma indirecta (relaciones `:B7_CHAIN` de salida)
- `chains`: la suma de `via` de sus relaciones `:B7_CHAIN` de salida


In [ ]:
# Comprobación del EJERCICIO 9: alcance de la ayuda
from collections import defaultdict

cadenas: list[Record] = driver.execute_query(
    "MATCH (a:B7Author)-[r:B7_CHAIN]->(c:B7Author) "
    "RETURN a.Id AS source, c.Id AS target, r.via AS via"
).records
respondio: list[Record] = driver.execute_query(
    "MATCH (a:B7Author)-[:B7_REPLIED_TO]->(b:B7Author) RETURN a.Id AS source, b.Id AS target"
).records
assert cadenas, ":B7_CHAIN está vacía: completa antes el ejercicio que la crea"

# Camino paralelo: recalculado en Python puro a partir de :B7_CHAIN y
# :B7_REPLIED_TO, que ya se validaron en ejercicios anteriores.
alcance: dict[int, int] = defaultdict(int)
cadenas_de: dict[int, int] = defaultdict(int)
for fila in cadenas:
    alcance[fila["source"]] += 1
    cadenas_de[fila["source"]] += fila["via"]
ayudados: dict[int, set[int]] = defaultdict(set)
for fila in respondio:
    ayudados[fila["source"]].add(fila["target"])
esperado: dict[int, tuple[int, int, int]] = {
    autor: (len(ayudados[autor]), alcance[autor], cadenas_de[autor]) for autor in alcance
}

resumen: list[Record] = driver.execute_query(
    "MATCH (:B7Bulletin {fingerprint: $fingerprint})-[r:B7_REACH]->(a:B7Author) "
    "RETURN a.Id AS author, r.helped AS helped, r.reach AS reach, r.chains AS chains",
    fingerprint=HUELLA,
).records
obtenido: dict[int, tuple[int, int, int]] = {
    fila["author"]: (fila["helped"], fila["reach"], fila["chains"]) for fila in resumen
}
print(f"Autores con alcance: esperados {len(esperado):,} | obtenidos {len(obtenido):,}")
assert len(resumen) == len(obtenido), "Hay más de una relación :B7_REACH para algún autor"
assert set(obtenido) == set(esperado), "Los autores con alcance no coinciden"
for autor, valores in esperado.items():
    assert obtenido[autor] == valores, f"Autor {autor}: helped, reach o chains no coinciden"
print("EJERCICIO 9: comprobado")

### EJERCICIO 10: resultado final Neo4j

Desde `:B7_REACH`, conecta el nodo de boletín con los `PARAMETROS.limite` autores cuya ayuda llega más lejos mediante `(:B7Bulletin)-[:B7_SUBMISSION]->(:User)`. Cada relación lleva:

- `position`: el puesto en el ranking, de 1 en adelante
- `helped`, `reach` y `chains`: copiados de la relación `:B7_REACH` de ese autor

Orden: `reach DESC`, `chains DESC`, `User.Id ASC`.

Para numerar el ranking, sigue el ejemplo «Numerar un ranking»: ordena con `LIMIT`, recoge las filas con `collect()` y recórrelas con `UNWIND range(...)`. Busca el nodo del boletín después, cuando ya tengas una fila por autor, para no tener que arrastrarlo por la agrupación.

Mira después el resultado y contesta en un comentario: ¿los autores que más ayudan directamente (`helped`) son también los que más lejos llegan? ¿Qué cambiaría en el ranking si `:B7_CHAIN` contara también las cadenas de tres ayudas?


In [ ]:
# Comprobación del EJERCICIO 10: resultado final Neo4j
alcance: list[Record] = driver.execute_query(
    "MATCH (:B7Bulletin {fingerprint: $fingerprint})-[r:B7_REACH]->(u:B7Author) "
    "RETURN u.Id AS userid, r.helped AS helped, r.reach AS reach, r.chains AS chains",
    fingerprint=HUELLA,
).records
assert alcance, ":B7_REACH está vacía: completa antes el ejercicio que la crea"

top: list[Record] = sorted(alcance, key=lambda f: (-f["reach"], -f["chains"], f["userid"]))[
    : PARAMETROS.limite
]

entrega: list[Record] = driver.execute_query(
    "MATCH (:B7Bulletin {fingerprint: $fingerprint})-[e:B7_SUBMISSION]->(u:User) "
    "RETURN e.position AS position, u.Id AS userid, e.helped AS helped, "
    "e.reach AS reach, e.chains AS chains ORDER BY e.position",
    fingerprint=HUELLA,
).records
print(f"userid esperado: {[f['userid'] for f in top]}")
print(f"userid obtenido: {[f['userid'] for f in entrega]}")
assert len(entrega) == len(top), f"Deberían ser {len(top)} filas"
assert [f["position"] for f in entrega] == list(range(1, len(entrega) + 1)), (
    "position debe ser un orden total 1..N sin huecos"
)
assert [f["userid"] for f in entrega] == [f["userid"] for f in top], (
    "No son los mismos autores, o no en el orden del enunciado"
)
for esperada, obtenida in zip(top, entrega):
    assert (esperada["helped"], esperada["reach"], esperada["chains"]) == (
        obtenida["helped"],
        obtenida["reach"],
        obtenida["chains"],
    ), f"Autor {esperada['userid']}: helped, reach o chains no coinciden con :B7_REACH"
print("EJERCICIO 10: comprobado")

## Resultado que se registrará

La consulta relee exclusivamente las relaciones `:B7_SUBMISSION`.

In [ ]:
import json
from decimal import Decimal

CONSULTA_FINAL: str = """
MATCH (b:B7Bulletin {fingerprint: $fingerprint})-[e:B7_SUBMISSION]->(u:User)
RETURN e.position AS position, u.Id AS userid, e.helped AS helped,
       e.reach AS reach, e.chains AS chains
ORDER BY e.position
"""

result: EagerResult = driver.execute_query(CONSULTA_FINAL, fingerprint=HUELLA)
RESULTADO_FINAL: list[dict[str, object]] = [dict(record) for record in result.records]


def _valor_json(value: object) -> int | float | str:
    if isinstance(value, Decimal):
        return int(value) if value == value.to_integral_value() else float(value)
    if hasattr(value, "isoformat"):
        return value.isoformat()
    raise TypeError(f"Valor no serializable: {type(value).__name__}")


RESULTADO_FINAL = json.loads(
    json.dumps(RESULTADO_FINAL, ensure_ascii=False, allow_nan=False, default=_valor_json)
)
if not RESULTADO_FINAL:
    print("Falta :B7_SUBMISSION: completa los ejercicios anteriores antes de registrar la entrega.")
else:
    for row in RESULTADO_FINAL:
        print(row)

## Entrega del resultado

El envío registra exactamente `RESULTADO_FINAL`, que se ha releído de la
estructura final de esta base de datos. Ejecuta esta celda una sola vez cuando
hayas terminado el boletín. Si la vuelves a ejecutar en el mismo kernel, no
creará una segunda entrega. Con el identificador de ejemplo no se contacta con
el servidor.

El cliente ya se descargó en «Preparación de la entrega», así que no hay
que copiar ningún ZIP junto al cuaderno.

In [ ]:
recibo_entrega: evidence.EvidenceReceipt | None
if not RESULTADO_FINAL:
    recibo_entrega = None
    print("No hay resultado final que enviar: completa los ejercicios anteriores.")
elif evidence.is_example_identifier(DNI):
    recibo_entrega = None
    print("DNI de ejemplo: no se ha enviado ninguna entrega. Cambia DNI antes de entregar.")
elif globals().get("recibo_entrega") is not None:
    print("La entrega ya se registró en este kernel; no se ha creado otra.")
else:
    recibo_entrega = evidence.submit_final_result_unless_example(
        course=CURSO,
        notebook=BOLETIN,
        student_identifier=DNI,
        query_text=CONSULTA_FINAL,
        result=RESULTADO_FINAL,
    )
recibo_entrega

## Comprobación de la entrega

Esta celda sólo consulta el último registro de este curso, boletín e
identificador. No vuelve a enviar `RESULTADO_FINAL` ni modifica SQLite, así que
puedes ejecutarla cuantas veces quieras: mientras no hagas un nuevo envío,
devolverá el mismo ID y la misma fecha UTC. El ID es el de la fila que el
profesor puede localizar en la base SQLite.

In [ ]:
estado_entrega: evidence.SubmissionStatus | None
if evidence.is_example_identifier(DNI):
    estado_entrega = None
    print("DNI de ejemplo: se omite la consulta de una entrega real.")
else:
    estado_entrega = evidence.check_submission_status(
        course=CURSO,
        notebook=BOLETIN,
        student_identifier=DNI,
    )
estado_entrega

In [ ]:
# @title Diapositiva: Para el examen: qué demuestra el resultado final
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #8a6a00;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#8a6a00" size="8" noshade style="display:none"><div style="color:#8a6a00;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#8a6a00" style="color:inherit"><b>Para el examen</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Para el examen: qué demuestra el resultado final</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>El resultado final depende de transformaciones materializadas anteriores</li><li style=\'margin:10px 0;list-style-position:outside\'>La granularidad elegida determina qué agregaciones son correctas</li><li style=\'margin:10px 0;list-style-position:outside\'>Los nulos y los desempates forman parte del contrato del resultado</li><li style=\'margin:10px 0;list-style-position:outside\'>El DNI parametriza los datos sin salir del cuaderno</li><li style=\'margin:10px 0;list-style-position:outside\'>Los ejercicios de rendimiento se justifican con planes, no con intuiciones</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #8a6a00;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pregunta típica: explicar qué relación lee el resultado final y qué recorrido la produjo</div></blockquote><hr color="#8a6a00" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Lo que deja la sesión 7
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Lo que deja la sesión 7</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Las <em>constraints</em> dan identidad e índice antes de cargar</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>UNWIND</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code> cargan por lotes sin duplicar</li><li style=\'margin:10px 0;list-style-position:outside\'>Cypher expresa patrones; <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>WITH</code> encadena etapas y la agrupación es implícita</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EXPLAIN</code> planifica y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>PROFILE</code> mide: el índice se comprueba en el plan</li><li style=\'margin:10px 0;list-style-position:outside\'>Una relación derivada abarata una pregunta frecuente</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>El valor del grafo está en los recorridos que hace baratos</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Cuatro modelos sobre los mismos datos
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Cuatro modelos sobre los mismos datos</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>MySQL: esquema y restricciones primero; el optimizador elige el plan</li><li style=\'margin:10px 0;list-style-position:outside\'>MongoDB: documentos; embeber o referenciar según la lectura</li><li style=\'margin:10px 0;list-style-position:outside\'>Cassandra: una tabla por consulta; la clave de partición decide dónde se busca</li><li style=\'margin:10px 0;list-style-position:outside\'>Neo4j: relaciones explícitas; los recorridos son la consulta</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Importa más comparar los paradigmas que memorizar la sintaxis de cada uno</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

## Guía de estudio para el examen

Antes del examen debes ser capaz de hacer esto **sin IA y sin este cuaderno delante**, usando las etiquetas `User`, `Question`, `Answer` y `Tag`:

- Escribir un `MATCH` con patrones de nodos y relaciones, y proyectar sólo las propiedades que necesitas con `RETURN`.
- Explicar qué hace `CREATE CONSTRAINT ... REQUIRE ... IS UNIQUE` y por qué crea también un índice.
- Explicar la diferencia entre `CREATE` y `MERGE`, y decir en qué casos `CREATE` produce duplicados que `MERGE` evita.
- Explicar por qué un `MERGE` con una propiedad variable en el patrón vuelve a duplicar, y corregirlo con `SET`, `ON CREATE SET` y `ON MATCH SET`.
- Numerar un ranking con `collect()` y `UNWIND range(...)`.
- Usar `UNWIND` para convertir una lista de mapas en filas que carguen o actualicen varios nodos en una sola consulta.
- Escribir agregaciones con `count()`, `collect()` y `ORDER BY`/`LIMIT`, y explicar por qué la agrupación es implícita.
- Distinguir `count(x)` de `count(DISTINCT x)` en un recorrido de varios saltos.
- Usar `WITH` para encadenar etapas y agregar resultados intermedios.
- Usar `OPTIONAL MATCH` para conservar nodos sin una relación concreta, y explicar qué pasaría con un `WHERE` mal colocado.
- Leer un `EXPLAIN` (plan) y un `PROFILE` (medida real), y decir si una consulta aprovecha un índice o hace un recorrido completo.
- Explicar por qué transformar una propiedad indexada (por ejemplo `toLower(t.TagName)`) le impide a Neo4j usar el índice directamente.
- Usar `shortestPath`/`allShortestPaths` y explicar qué significa el número de saltos que devuelven.
- Explicar por qué Neo4j no admite propiedades con valor `null`, y qué hay que hacer al cargar filas con campos ausentes.
- Justificar cuándo compensa materializar una relación derivada y quién se encarga de mantenerla.

Si al intentar cualquiera de estos puntos te quedas en blanco, vuelve al ejercicio o al ejemplo correspondiente de este cuaderno antes del examen.

## Ampliación no evaluable: análisis de influencia y expertos por dominio

Como posible trabajo abierto, se puede identificar "expertos influyentes" en diferentes dominios (tags). Un experto influyente es alguien que:

1. Tiene alta actividad en un tag específico (muchas preguntas/respuestas)
2. Recibe interacción positiva (upvotes, respuestas aceptadas)
3. Está bien conectado en la red social (muchas relaciones RECIPROCATE)
4. Sus respuestas generan "cascadas" de actividad (comentarios, más respuestas)

Se pide:

- **Calcular el "Score de Expertise" por Tag**, donde esos cuatro puntos de arriba se combinan en un único score, con pesos a definir por vosotros.

El grafo de esta sesión sólo carga `Posts` (y los `Users` que los escriben): `Votes.parquet` y `Comments.parquet` existen en el release, pero habría que cargarlos antes de usar votos o comentarios.
